In [48]:
import numpy as np
import pandas as pd



import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/hoangvuasoclo/profile-core/synthetic_output.json
/kaggle/input/datasets/hoangvuasoclo/profile-core/profile_core.json
/kaggle/input/datasets/hoangvuasoclo/raw-data/raw_data_cleaned.json
/kaggle/input/datasets/hoangvuasoclo/generation-style/narrative_style_registry.json
/kaggle/input/datasets/hoangvuasoclo/pii-framework/pii_schema.json


In [49]:
# =============================================================================
# CELL 0 — CÀI THƯ VIỆN
# =============================================================================
# Chạy 1 lần đầu phiên Kaggle. Sau khi chạy CẦN RESTART KERNEL.
#
# Lưu ý:
# - Kaggle image (2025/2026) KHÔNG có sẵn bitsandbytes → phải cài.
# - transformers 4.57+ yêu cầu sentence-transformers >= 3.0 (vì
#   transformers đã bỏ class PreTrainedConfig cũ).
# =============================================================================

import subprocess
import sys

def pip_install(pkgs, upgrade=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if upgrade:
        cmd.append("--upgrade")
    cmd.extend(pkgs)
    subprocess.check_call(cmd)


pip_install([
    "bitsandbytes>=0.43.0",
    "accelerate>=0.34.0",
])

pip_install(["transformers>=4.51.0"], upgrade=True)


pip_install(["sentence-transformers>=3.0.0"], upgrade=True)

# ---- Verify imports ----
print()
try:
    import bitsandbytes as bnb
    print(f"✅ bitsandbytes          {bnb.__version__}")
except Exception as e:
    print(f"❌ bitsandbytes          FAILED: {e}")

try:
    import transformers
    print(f"✅ transformers          {transformers.__version__}")
except Exception as e:
    print(f"❌ transformers          FAILED: {e}")

try:
    import accelerate
    print(f"✅ accelerate            {accelerate.__version__}")
except Exception as e:
    print(f"❌ accelerate            FAILED: {e}")

try:
    import sentence_transformers
    print(f"✅ sentence-transformers {sentence_transformers.__version__}")
except Exception as e:
    print(f"❌ sentence-transformers FAILED: {e}")

import torch
print(f"✅ torch                 {torch.__version__}  | CUDA: {torch.cuda.is_available()}  | GPUs: {torch.cuda.device_count()}")

print()
print("=" * 64)
print("⚠️  QUAN TRỌNG — HÃY RESTART KERNEL SAU CELL NÀY")
print("    (Run → Restart & Clear Cell Outputs)")
print("    Vì transformers + sentence-transformers đã upgrade —")
print("    phải reload module mới. Sau khi restart, chạy lại từ Cell 1.")
print("=" * 64)


✅ bitsandbytes          0.49.2
✅ transformers          5.9.0
✅ accelerate            1.13.0
✅ sentence-transformers 5.5.1
✅ torch                 2.10.0+cu128  | CUDA: True  | GPUs: 2

⚠️  QUAN TRỌNG — HÃY RESTART KERNEL SAU CELL NÀY
    (Run → Restart & Clear Cell Outputs)
    Vì transformers + sentence-transformers đã upgrade —
    phải reload module mới. Sau khi restart, chạy lại từ Cell 1.


# SecurePrep Kaggle Full Pipeline v2

Self-contained Kaggle notebook for the updated SecurePrep generation pipeline.

Updates:
- Uses `profile_core/profile_core.json` as seed bank, not `synthetic_output.json`.
- Adds required input `narrative_style_registry.json`.
- Uses `injection_list.fields[]` as source of truth.
- `context_seeds` are not annotated.
- `locked_values` are derived from annotated fields.
- Final `spans` are exact offsets in final text.
- Noise is post-B11 only and rolls back if validation fails.
- Checkpoint and failure logs are JSONL files suitable for Kaggle Dataset sync.

In [50]:
# CELL 1 - USER CONFIG
PERSON_ID = 0
FORM_ID_START_OVERRIDE = 2005
FORM_ID_END_OVERRIDE = 2025
ROWS_PER_FORM = 1
FORMS_PER_ACCOUNT = 10

# Single-GPU mode. Keep shard_count=1 for one process over the whole range.
PARALLEL_SHARD_COUNT = 1
PARALLEL_SHARD_INDEX = 0
# Pin to Kaggle's first visible GPU. Use 1 only if you explicitly want the second T4.
GPU_DEVICE_ID = 0
# Set True to run GPU0 + GPU1 in parallel threads with the same Gemma model.
AUTO_GPU_WORKERS = True
AUTO_GPU_WORKER_COUNT = 2

# Old input dataset root.
DATA_DIR = "/kaggle/input/datasets/hoangvuasoclo"

# narrative_style_registry.json is stored inside the same Kaggle input dataset.
STYLE_DATA_DIR = f"{DATA_DIR}/generation-style"

PATH_PROFILE_CORE = f"{DATA_DIR}/profile-core/profile_core.json"
PATH_RAW_DATA = f"{DATA_DIR}/raw-data/raw_data_cleaned.json"
PATH_PII_SCHEMA = f"{DATA_DIR}/pii-framework/pii_schema.json"

WORK_DIR = "/kaggle/working"
OUTPUT_DIR = f"{WORK_DIR}/output"
CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
KAGGLE_CHECKPOINT_DATASET = "hoangvuasoclo/secureprep-ckpts-v2"

MODEL_NAME = "google/gemma-3-4b-it"
USE_MOCK_BACKEND = False
LOAD_IN_4BIT = True
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 1600
MAX_ATTEMPTS = 3
NOISE_PROBABILITY = 0.0
MIN_CHARS = 120
SEED = 20260525
TIME_LIMIT_S = 38_000
# Local checkpoint writes to /kaggle/working/checkpoints.
# Keep this at 1 when debugging: every completed/failed row is saved immediately.
CHECKPOINT_EVERY_ROWS = 1
EXPORT_RECORDS_JSON = True
OVERWRITE_SAME_RANGE_CHECKPOINT = True
LOCKED_FIELDS_MIN = 8
LOCKED_FIELDS_MAX = 18
FORM_WORD_MIN_RATIO = 1.10
FORM_WORD_MAX_RATIO = 1.50
MIN_WORDS_PER_LOCKED_VALUE = 25
TARGET_MAX_WORDS = 800

# External checkpoint pushes /kaggle/working/checkpoints to KAGGLE_CHECKPOINT_DATASET.
# Set to 1 only if you really want a Kaggle Dataset version after every row.
KAGGLE_PUSH_EVERY_ROWS = 1
PRINT_GENERATED_RECORDS = False
PRINT_RECORD_LIMIT = 1
FAIL_FAST_ON_ERROR = True

print("Config loaded", DATA_DIR, STYLE_DATA_DIR, MODEL_NAME)

Config loaded /kaggle/input/datasets/hoangvuasoclo /kaggle/input/datasets/hoangvuasoclo/generation-style google/gemma-3-4b-it


In [51]:
# CELL 2 - IMPORTS, IO, KAGGLE CHECKPOINT SYNC
import os, re, gc, json, time, random, shutil, hashlib, tempfile, threading, traceback, subprocess, unicodedata, math
from pathlib import Path
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from collections import Counter

EFFECTIVE_GPU_DEVICE_ID = GPU_DEVICE_ID
if EFFECTIVE_GPU_DEVICE_ID is None and PARALLEL_SHARD_COUNT > 1:
    EFFECTIVE_GPU_DEVICE_ID = PARALLEL_SHARD_INDEX
# CUDA_VISIBLE_DEVICES is NOT set here so both GPUs remain visible for threading.
# Each worker pins its model via device_map={"": gpu_id} instead.
print("GPU worker config:", "shard", PARALLEL_SHARD_INDEX, "of", PARALLEL_SHARD_COUNT, "effective_gpu_id", EFFECTIVE_GPU_DEVICE_ID)

def print_gpu_environment():
    try:
        import torch
        names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []
        print("CUDA available:", torch.cuda.is_available(), "visible_gpus:", torch.cuda.device_count(), "names:", names)
    except Exception as e:
        print("CUDA environment check failed:", e)

if not AUTO_GPU_WORKERS:
    print_gpu_environment()

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def write_json_atomic(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    tmp.replace(path)

def append_jsonl(path, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def resolve_existing_path(candidates):
    for p in candidates:
        if p and os.path.exists(p):
            return p
    raise FileNotFoundError("\n".join(map(str, candidates)))

PATH_STYLE_REGISTRY = resolve_existing_path([
    f"{STYLE_DATA_DIR}/narrative_style_registry.json",
    f"{STYLE_DATA_DIR}/generation_style/narrative_style_registry.json",
    f"{DATA_DIR}/generation-style/narrative_style_registry.json",
    f"{DATA_DIR}/generation_style/narrative_style_registry.json",
])

for p in [PATH_PROFILE_CORE, PATH_RAW_DATA, PATH_PII_SCHEMA, PATH_STYLE_REGISTRY]:
    print("FOUND" if os.path.exists(p) else "MISSING", p)

_KAGGLE_API_READY = False
_PUSH_LOCK = threading.Lock()

def setup_kaggle_api():
    global _KAGGLE_API_READY
    if not KAGGLE_CHECKPOINT_DATASET:
        print("Kaggle checkpoint push disabled")
        return
    try:
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        kdir = Path.home() / ".kaggle"
        kdir.mkdir(parents=True, exist_ok=True)
        cred = kdir / "kaggle.json"
        cred.write_text(json.dumps({
            "username": s.get_secret("KAGGLE_USERNAME"),
            "key": s.get_secret("KAGGLE_KEY"),
        }), encoding="utf-8")
        os.chmod(cred, 0o600)
        _KAGGLE_API_READY = True
        print("Kaggle API ready:", KAGGLE_CHECKPOINT_DATASET)
    except Exception as e:
        print("Kaggle API setup skipped:", e)

def write_dataset_metadata():
    if KAGGLE_CHECKPOINT_DATASET:
        write_json_atomic(Path(CHECKPOINT_DIR) / "dataset-metadata.json", {
            "id": KAGGLE_CHECKPOINT_DATASET,
            "licenses": [{"name": "other"}],
            "title": "SecurePrep Checkpoints V2",
        })

def sync_existing_checkpoints_from_kaggle():
    if not _KAGGLE_API_READY or not KAGGLE_CHECKPOINT_DATASET:
        return
    tmp_dir = tempfile.mkdtemp(prefix="secureprep_ckpt_sync_")
    try:
        r = subprocess.run(
            ["kaggle", "datasets", "download", "-d", KAGGLE_CHECKPOINT_DATASET, "-p", tmp_dir, "--unzip"],
            capture_output=True, text=True, timeout=180,
        )
        if r.returncode != 0:
            print("Checkpoint sync skipped:", r.stderr[:300])
            return
        copied = 0
        for src in Path(tmp_dir).glob("**/*"):
            if src.is_file() and src.name != "dataset-metadata.json":
                dst = Path(CHECKPOINT_DIR) / src.relative_to(tmp_dir)
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(src, dst)
                    copied += 1
        print("Synced checkpoint files:", copied)
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def push_checkpoint_to_kaggle(message="checkpoint", wait=False):
    if not _KAGGLE_API_READY or not KAGGLE_CHECKPOINT_DATASET:
        return
    if not _PUSH_LOCK.acquire(blocking=wait):
        print("Kaggle push skipped: another push is running")
        return
    try:
        write_dataset_metadata()
        r = subprocess.run(
            ["kaggle", "datasets", "version", "-p", CHECKPOINT_DIR, "-m", message, "--dir-mode", "zip"],
            capture_output=True, text=True, timeout=240,
        )
        if r.returncode == 0:
            print("Pushed checkpoint:", message)
        else:
            print("Kaggle push failed:", "returncode=", r.returncode)
            if r.stdout:
                print("kaggle stdout:", r.stdout[:1000])
            if r.stderr:
                print("kaggle stderr:", r.stderr[:1000])
    except Exception as e:
        print("Kaggle push error:", e)
    finally:
        _PUSH_LOCK.release()

def push_checkpoint_async(message="checkpoint"):
    threading.Thread(target=push_checkpoint_to_kaggle, args=(message,), daemon=True).start()

setup_kaggle_api()
sync_existing_checkpoints_from_kaggle()

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = HF_TOKEN or UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    if not HF_TOKEN:
        print("HF_TOKEN not found in Kaggle Secrets/env:", e)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded")

GPU worker config: shard 0 of 1 effective_gpu_id 0
FOUND /kaggle/input/datasets/hoangvuasoclo/profile-core/profile_core.json
FOUND /kaggle/input/datasets/hoangvuasoclo/raw-data/raw_data_cleaned.json
FOUND /kaggle/input/datasets/hoangvuasoclo/pii-framework/pii_schema.json
FOUND /kaggle/input/datasets/hoangvuasoclo/generation-style/narrative_style_registry.json
Kaggle API ready: hoangvuasoclo/secureprep-ckpts-v2
Synced checkpoint files: 0
HF_TOKEN loaded


In [52]:
# CELL 3 - LOAD INPUTS
raw_records = load_json(PATH_RAW_DATA)
profile_core = load_json(PATH_PROFILE_CORE)
pii_schema = load_json(PATH_PII_SCHEMA)
style_registry = load_json(PATH_STYLE_REGISTRY)

print("raw_records", len(raw_records))
print("profile_core keys", list(profile_core)[:12])
print("pii_fields", len(pii_schema.get("fields") or []))
print("style_registry", style_registry.get("version"), "voices", len(style_registry.get("voices") or []))

required = ["template_id", "record_type", "field", "sub_field", "agency", "procedure_raw_text", "attached_forms"]
print("missing_raw", {k: sum(1 for r in raw_records if not r.get(k)) for k in required})
if not isinstance(profile_core, dict) or "ho" not in profile_core or "dia_danh" not in profile_core:
    raise ValueError("profile_core.json is not the expected seed bank")

raw_records 9020
profile_core keys ['_meta', 'ho', 'ten', 'ethnic_names', 'dia_danh', 'extended_places', 'vung_sample', 'vung_weights', 'nghe_nghiep', 'dan_toc', 'ton_giao', 'hon_nhan_by_age']
pii_fields 27
style_registry 2026-05-27 voices 25
missing_raw {'template_id': 0, 'record_type': 0, 'field': 0, 'sub_field': 0, 'agency': 0, 'procedure_raw_text': 0, 'attached_forms': 0}


In [53]:
# CELL 4 - PROFILE, STYLE, INJECTION
@dataclass
class InjectionField:
    name: str
    value: str
    label: str
    sensitivity: str
    identifier_type: str
    context_dependency: str
    source: str = "profile_fake"
    annotate: bool = True
    level: str | None = None
    name_vi: str | None = None
    law_ref: str | None = None
    value_status: str | None = None

@dataclass
class ContextSeed:
    name: str
    value: str
    source: str = "profile_fake"
    annotate: bool = False

def stable_digits(text, n):
    digest = hashlib.sha256(str(text).encode("utf-8")).hexdigest()
    return "".join(str(int(ch, 16) % 10) for ch in digest)[:n]

def weighted_choice(weights, rng):
    items = []
    for key, value in weights.items():
        weight = value.get("weight", 1.0) if isinstance(value, dict) else value
        try:
            weight = float(weight)
        except Exception:
            weight = 1.0
        items.append((key, max(weight, 0.0)))
    total = sum(w for _, w in items)
    point = rng.random() * total if total > 0 else 0
    acc = 0.0
    for key, weight in items:
        acc += weight
        if acc >= point:
            return key
    return items[-1][0]

def ascii_slug(text):
    text = text.lower().replace("đ", "d")
    normalized = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in normalized if ch.isascii() and ch.isalnum())

def make_phone(core, rng):
    prefixes = []
    for value in (core.get("phone_prefix_by_carrier") or {}).values():
        if isinstance(value, list):
            prefixes.extend(value)
    prefix = rng.choice(prefixes or ["09"])
    return prefix + "".join(str(rng.randint(0, 9)) for _ in range(10 - len(prefix)))

def make_profile_fake(core, index):
    rng = random.Random(SEED + index)
    gender = rng.choice(["Nam", "Nữ"])
    name_key = "Nam" if gender == "Nam" else "Nu"
    last = weighted_choice(core.get("ho") or {"Nguyễn": 1}, rng)
    given = rng.choice((core.get("ten") or {}).get(name_key) or ["Văn An"])
    full_name = f"{last} {given}"
    year = weighted_choice(core.get("birth_year_weights") or {"1995": 1}, rng)
    dob = f"{rng.randint(1,28):02d}/{rng.randint(1,12):02d}/{year}"
    places = core.get("dia_danh") or {}
    province_code = rng.choice(list(places)) if places else "001"
    place = places.get(province_code) or {"ten": "Hà Nội", "huyen": ["Cầu Giấy"], "xa_phuong": ["Dịch Vọng"]}
    address = f"Phường/Xã {rng.choice(place.get('xa_phuong') or ['Dịch Vọng'])}, Quận/Huyện {rng.choice(place.get('huyen') or ['Cầu Giấy'])}, Tỉnh/Thành phố {place.get('ten') or 'Hà Nội'}"
    domains = core.get("email_domains") or {"personal": ["example.vn"]}
    flat_domains = [d for values in domains.values() for d in values] if isinstance(domains, dict) else list(domains)
    profile = {
        "profile_id": f"profile_{index:06d}",
        "full_name": full_name,
        "dob": dob,
        "birth_year": str(year),
        "gender": gender,
        "phone": make_phone(core, rng),
        "cccd": province_code[:3].rjust(3, "0") + stable_digits(full_name + dob + str(index), 9),
        "address": address,
        "nationality": "Việt Nam",
        "marital_status": rng.choice(["Độc thân", "Đã kết hôn", "Ly hôn", "Góa"]),
        "email": f"{ascii_slug(full_name)}{stable_digits(full_name + str(index), 4)}@{rng.choice(flat_domains or ['example.vn'])}",
        "passport_number": f"P{stable_digits(full_name + 'passport', 7)}",
        "occupation": weighted_choice(core.get("nghe_nghiep") or {"Người nộp hồ sơ": 1}, rng),
        "ethnicity": weighted_choice(core.get("dan_toc") or {"Kinh": 1}, rng),
        "religion": weighted_choice(core.get("ton_giao") or {"Không tôn giáo": 1}, rng),
        "source_profile": {"source": "profile_core", "index": index},
    }
    return add_legal_candidate_attributes(profile, index)

def add_legal_candidate_attributes(profile, index):
    enriched = dict(profile)
    full_name = str(enriched.get("full_name") or f"Nguoi dung {index:06d}")
    base = full_name + str(index)
    slug = ascii_slug(full_name) or f"user{index:06d}"
    enriched.setdefault("driver_license", "GPLX" + stable_digits(base + "driver", 9))
    enriched.setdefault("vehicle_plate", f"{stable_digits(base + 'plate', 2)}A-{stable_digits(base + 'plate2', 5)}")
    enriched.setdefault("bank_account", stable_digits(base + "bank", 12))
    enriched.setdefault("digital_account", f"{slug}.{stable_digits(base + 'acct', 3)}")
    enriched.setdefault("eid_credentials", f"VNeID-{stable_digits(base + 'eid', 10)}")
    enriched.setdefault("location_data", f"vị trí thường trú mã {stable_digits(base + 'loc', 6)}")
    enriched.setdefault("family_relations", f"người liên hệ khẩn cấp là thân nhân của {full_name}")
    statuses = ["có tiền sử dị ứng thuốc", "đang theo dõi huyết áp", "cần hỗ trợ đi lại khi làm thủ tục", "đang điều trị bệnh mạn tính"]
    enriched.setdefault("health_status", statuses[index % len(statuses)])
    enriched.setdefault("criminal_record", "không có án tích theo phiếu lý lịch tư pháp")
    enriched.setdefault("biometric", f"mã tham chiếu sinh trắc học BIO-{stable_digits(base + 'bio', 8)}")
    enriched.setdefault("political_view", "không khai báo quan điểm chính trị trong hồ sơ")
    enriched.setdefault("private_life", "thông tin đời sống riêng tư trong phần giải trình")
    enriched.setdefault("sexual_orientation", "không công khai xu hướng tính dục")
    enriched.setdefault("behavioral_data", "lịch sử truy cập cổng dịch vụ công vào buổi tối")
    return enriched

FIELD_KEYWORDS = {
    "full_name": ["họ và tên", "họ tên", "tên người", "người đại diện", "người nộp"],
    "dob": ["ngày sinh", "năm sinh", "sinh ngày"],
    "gender": ["giới tính", "nam/nữ"],
    "cccd": ["cccd", "cmnd", "căn cước", "định danh cá nhân", "số định danh"],
    "phone": ["số điện thoại", "điện thoại", "telephone", "phone"],
    "email": ["email", "e-mail", "thư điện tử"],
    "address": ["địa chỉ", "nơi cư trú", "thường trú", "tạm trú", "quê quán"],
    "passport_number": ["hộ chiếu", "passport"],
    "nationality": ["quốc tịch"],
    "marital_status": ["hôn nhân", "tình trạng hôn nhân"],
    "ethnicity": ["dân tộc"],
    "religion": ["tôn giáo"],
    "bank_account": ["tài khoản ngân hàng", "số tài khoản", "ngân hàng"],
    "driver_license": ["giấy phép lái xe", "gplx", "bằng lái"],
    "vehicle_plate": ["biển số", "biển kiểm soát", "đăng ký xe"],
    "health_status": ["sức khỏe", "bệnh", "khuyết tật", "y tế"],
}

def parse_form_blank(raw_record, max_contexts=40):
    forms = raw_record.get("attached_forms") or []
    text = forms[0].get("form_raw_text") if forms and isinstance(forms[0], dict) else ""
    compact = " ".join(str(text or raw_record.get("procedure_raw_text") or "").split())
    norm = compact.lower()
    placeholder_fields = []
    for match in re.finditer(r"\[PLACEHOLDER\]|_{2,}", compact):
        context = compact[max(0, match.start() - 90): min(len(compact), match.end() + 90)]
        hits = [field for field, kws in FIELD_KEYWORDS.items() if any(kw in context.lower() for kw in kws)]
        placeholder_fields.append({"placeholder": match.group(0), "start": match.start(), "context": context, "matched_fields": hits})
        if len(placeholder_fields) >= max_contexts:
            break
    context_hints, all_hits = [], []
    for field, kws in FIELD_KEYWORDS.items():
        for kw in kws:
            if kw in norm:
                context_hints.append({"field": field, "keyword": kw})
                all_hits.append(field)
                break
    required = sorted(set(f for item in placeholder_fields for f in item["matched_fields"]))
    optional = sorted(set(all_hits) - set(required))
    return {
        "raw_without_placeholder": re.sub(r"\[PLACEHOLDER\]", "", compact)[:2000],
        "placeholder_fields": placeholder_fields,
        "required_fields": required,
        "optional_fields": optional,
        "context_hints": context_hints,
        "matched_fields": sorted(set(required + optional)),
        "placeholder_count": len(placeholder_fields),
    }

def enrich_profile_by_form(profile, form_blank, index):
    enriched = dict(profile)
    needed = set(form_blank.get("matched_fields") or [])
    base = enriched.get("full_name", "") + str(index)
    if "bank_account" in needed and not enriched.get("bank_account"):
        enriched["bank_account"] = stable_digits(base + "bank", 12)
    if "driver_license" in needed and not enriched.get("driver_license"):
        enriched["driver_license"] = "GPLX" + stable_digits(base + "driver", 9)
    if "vehicle_plate" in needed and not enriched.get("vehicle_plate"):
        enriched["vehicle_plate"] = f"{stable_digits(base + 'plate', 2)}A-{stable_digits(base + 'plate2', 5)}"
    if "health_status" in needed and not enriched.get("health_status"):
        statuses = ["có tiền sử dị ứng thuốc", "đang theo dõi huyết áp", "cần hỗ trợ đi lại"]
        enriched["health_status"] = statuses[index % len(statuses)]
    return enriched

PII_META = {f.get("field"): f for f in (pii_schema.get("fields") or []) if isinstance(f, dict) and f.get("field")}
PREFERRED_FIELDS = ["full_name", "dob", "cccd", "phone", "email", "address", "passport_number", "driver_license", "vehicle_plate", "bank_account", "digital_account", "eid_credentials", "gender", "nationality", "marital_status", "ethnicity", "religion", "health_status", "location_data", "family_relations", "criminal_record", "biometric", "private_life", "behavioral_data", "political_view", "sexual_orientation"]
SPI_FIELD_NAMES = {"ethnicity", "political_view", "religion", "private_life", "health_status", "biometric", "sexual_orientation", "criminal_record", "location_data", "eid_credentials", "bank_account", "behavioral_data"}
REGEX_LIKE_FIELDS = {"phone", "email", "cccd", "passport_number", "driver_license", "vehicle_plate", "bank_account", "eid_credentials", "digital_account"}
_NON_LOCKABLE_FIELDS = {"gender", "nationality"}

def pick_id(items, rng, fallback):
    return str((rng.choice(items) if items else {}).get("id") or fallback)

def _registry_items_by_id(registry, key):
    return {str(item.get("id")): item for item in (registry.get(key) or []) if isinstance(item, dict) and item.get("id")}

def _pick_registry_item(items, rng, fallback_id):
    if not items:
        return {"id": fallback_id}
    return dict(rng.choice(items))

def _items_by_allowed_ids(items, allowed_ids):
    allowed = set(allowed_ids or [])
    return [item for item in items if item.get("id") in allowed] if allowed else list(items)

def _preferred_by_context(registry, raw_record, key, all_items):
    text = " ".join(str(raw_record.get(k) or "") for k in ["record_type", "field", "sub_field", "agency", "procedure_raw_text"]).lower()
    preferred_ids = []
    for rule in registry.get("compatibility_rules") or []:
        keywords = [str(x).lower() for x in rule.get("if_record_context_contains", [])]
        if keywords and any(keyword in text for keyword in keywords):
            preferred_ids.extend(rule.get(f"prefer_{key}", []) or [])
    by_id = {item.get("id"): item for item in all_items}
    return [by_id[item_id] for item_id in preferred_ids if item_id in by_id]

def _compact_style_item(item):
    return {
        k: item.get(k)
        for k in ["id", "label", "description", "family", "intensity", "paragraph_count"]
        if item.get(k) is not None
    }

def _difficulty_factor_details(registry, difficulty_id, rng, seed_text):
    policy = registry.get("difficulty_policy") or {}
    by_id = {item.get("id"): item for item in policy.get("factor_pool", []) if isinstance(item, dict)}
    candidate_ids = list((policy.get("level_to_factor_ids") or {}).get(difficulty_id) or [])
    if not candidate_ids:
        return []
    selection = policy.get("selection") or {}
    min_map = selection.get("min_factors_per_record") or {}
    max_map = selection.get("max_factors_per_record") or {}
    lo = int(min_map.get(difficulty_id, 1))
    hi = int(max_map.get(difficulty_id, min(3, len(candidate_ids))))
    hi = max(lo, min(hi, len(candidate_ids)))
    count = rng.randint(lo, hi)
    rng.shuffle(candidate_ids)
    selected = []
    blocked_pairs = [set(pair) for pair in selection.get("do_not_select_together", [])]
    for factor_id in candidate_ids:
        if factor_id not in by_id:
            continue
        current = {item.get("id") for item in selected}
        if any(factor_id in pair and current.intersection(pair) for pair in blocked_pairs):
            continue
        selected.append(by_id[factor_id])
        if len(selected) >= count:
            break
    return [_compact_style_item(item) | {"effects": item.get("effects")} for item in selected]

def make_format_plan(registry, rng, raw_record=None):
    raw_record = raw_record or {}
    record_tags = registry.get("record_tags") or []
    voices = registry.get("voices") or []
    tones = registry.get("tones") or []
    structures = registry.get("structures") or []
    styles = registry.get("styles") or []
    difficulties = registry.get("difficulty_levels") or []

    preferred_styles = _preferred_by_context(registry, raw_record, "styles", styles)
    style = _pick_registry_item(preferred_styles or styles, rng, "administrative_case_note")
    compatible_voices = _items_by_allowed_ids(voices, style.get("compatible_voices"))
    compatible_tones = _items_by_allowed_ids(tones, style.get("compatible_tones"))
    compatible_structures = _items_by_allowed_ids(structures, style.get("compatible_structures"))
    voice = _pick_registry_item(compatible_voices or voices, rng, "nguoi_nop_ho_so")
    tone = _pick_registry_item(compatible_tones or tones, rng, "hanh_chinh_trung_tinh")
    structure = _pick_registry_item(compatible_structures or structures, rng, "mot_doan_dai")
    record_tag = _pick_registry_item(record_tags, rng, "admin_form")
    difficulty = _pick_registry_item(difficulties, rng, "medium")
    factor_details = _difficulty_factor_details(registry, difficulty.get("id"), rng, str(raw_record.get("record_index") or raw_record.get("template_id") or raw_record.get("record_type")))
    return {
        "record_tag": record_tag.get("id"),
        "voice": voice.get("id"),
        "tone": tone.get("id"),
        "structure": structure.get("id"),
        "style": style.get("id"),
        "difficulty": difficulty.get("id"),
        "difficulty_factors": [item.get("id") for item in factor_details],
        "difficulty_factor_details": factor_details,
        "record_tag_detail": _compact_style_item(record_tag),
        "voice_detail": _compact_style_item(voice),
        "tone_detail": _compact_style_item(tone),
        "structure_detail": _compact_style_item(structure),
        "style_detail": _compact_style_item(style),
        "difficulty_detail": _compact_style_item(difficulty),
        "registry_version": registry.get("version"),
    }

def locked_target_for(difficulty, word_target=180):
    if word_target <= 180:
        low, high = 5, 8
    elif word_target <= 450:
        low, high = 10, 14
    elif word_target <= 800:
        low, high = 14, 18
    else:
        low, high = 16, 22
    if difficulty in {"hard", "adversarial", "eval_hard"}:
        low, high = max(low, 14), max(high, 18)
    elif difficulty == "easy":
        high = min(high, max(low, 10))
    low = max(LOCKED_FIELDS_MIN, low)
    high = min(LOCKED_FIELDS_MAX, high)
    return min(low, high), max(low, high)

def attribute_target_for(difficulty, word_target=350):
    if word_target <= 180:
        low, high = 5, 9
    elif word_target <= 450:
        low, high = 10, 15
    elif word_target <= 800:
        low, high = 15, 22
    else:
        low, high = 18, 25
    if difficulty in {"hard", "eval_hard"}:
        low, high = max(low, 18), max(high, 25)
    elif difficulty == "adversarial":
        low, high = max(low, 14), max(high, 20)
    elif difficulty == "easy":
        high = min(high, max(low, 12))
    return low, high

def estimate_word_target(form_blank):
    raw = str(form_blank.get("raw_without_placeholder") or "")
    words = raw.split()
    return min(800, max(120, int(len(words) * 1.25))) if words else 350

def build_injection_list(profile, rng, difficulty, word_target=180, selected_attributes=None):
    fields = []
    for name in (selected_attributes or PREFERRED_FIELDS):
        value = profile.get(name)
        meta = PII_META.get(name, {})
        if value:
            fields.append(InjectionField(name, str(value), str(meta.get("label") or meta.get("sensitivity") or "PII"),
                                         str(meta.get("sensitivity") or "PII"), str(meta.get("identifier_type") or "direct"),
                                         str(meta.get("context_dependency") or "context_free"), annotate=name not in _NON_LOCKABLE_FIELDS, level=meta.get("level"), name_vi=meta.get("name_vi"), law_ref=meta.get("law_ref")))
    low, high = locked_target_for(difficulty, word_target)
    target = min(len(fields), rng.randint(low, high))
    if selected_attributes:
        selected = fields
    else:
        required = {"full_name", "cccd", "phone"}
        selected = [f for f in fields if f.name in required]
        rest = [f for f in fields if f.name not in required]
        rng.shuffle(rest)
        selected = (selected + rest)[:target]
    seeds = [ContextSeed(k, str(profile[k])) for k in ["occupation", "birth_year", "gender"] if profile.get(k)]
    return {"fields": [asdict(f) for f in selected], "context_seeds": [asdict(s) for s in seeds], "locked_values": [f.value for f in selected if f.annotate]}

def build_event_background(profile, info, form_blank, format_plan):
    tag = format_plan.get("record_tag") or "admin_form"
    channel = {"citizen_message": "tin nhắn", "service_chat": "trao đổi hỗ trợ", "email_supplement": "email", "officer_note": "ghi chú nội bộ"}.get(tag, "bộ phận một cửa")
    record_type = info.get("record_type") or "hồ sơ hành chính"
    agency = info.get("agency") or "cơ quan tiếp nhận"
    suggested = form_blank.get("matched_fields") or ["full_name", "cccd", "phone"]
    background_variants = [
        f"Một {profile.get('occupation')} sinh năm {profile.get('birth_year')} cần bổ sung thông tin cho thủ tục {record_type}. Hồ sơ được xử lý qua {channel} tại {agency}, nội dung tập trung vào các trường {', '.join(suggested[:5])}.",
        f"Trong phần rà soát biểu mẫu {record_type}, người nộp hồ sơ cần làm rõ các trường {', '.join(suggested[:5])}. Kênh phát sinh là {channel}, cơ quan xử lý là {agency}.",
        f"Hồ sơ {record_type} có yêu cầu đối chiếu thông tin cá nhân của một {profile.get('occupation')} sinh năm {profile.get('birth_year')}. Việc trao đổi diễn ra qua {channel} với {agency}.",
        f"Người nộp hồ sơ gửi bổ sung thông tin liên quan đến {record_type}, chủ yếu quanh {', '.join(suggested[:5])}. Bối cảnh xử lý gắn với {agency} và kênh {channel}.",
    ]
    background = background_variants[int(info.get("record_index") or 0) % len(background_variants)]
    return {
        "event_list": [{"event_id": "evt_01", "actor_role": "nguoi_nop_ho_so", "procedure_goal": f"thực hiện hoặc bổ sung hồ sơ: {record_type}", "trigger": "hồ sơ cần đối chiếu thông tin cá nhân theo biểu mẫu", "channel": channel, "agency_context": agency, "document_state": "đang chuẩn bị hoặc bổ sung", "allowed_context_entities": [agency, channel]}],
        "background_context": background,
        "record_tags": [tag],
        "context_hint": {"suggested_fields": suggested, "forbidden_new_pii": True},
    }

def build_content_plan(profile, form_blank, event_background, format_plan, rng):
    word_target = estimate_word_target(form_blank)
    low, high = attribute_target_for(format_plan.get("difficulty") or "medium", word_target)
    target = rng.randint(low, high)
    form_fields = [f for f in form_blank.get("matched_fields", []) if profile.get(f)]
    required = []
    for name in form_fields:
        if name not in required:
            required.append(name)
    if not required and profile.get("full_name"):
        required.append("full_name")
    candidates = []
    optional = [name for name in PREFERRED_FIELDS if name not in required and profile.get(name)]
    spi_optional = [name for name in optional if name in SPI_FIELD_NAMES]
    pii_optional = [name for name in optional if name not in SPI_FIELD_NAMES and name not in REGEX_LIKE_FIELDS]
    regex_optional = [name for name in optional if name not in SPI_FIELD_NAMES and name in REGEX_LIKE_FIELDS]
    rng.shuffle(spi_optional)
    rng.shuffle(pii_optional)
    rng.shuffle(regex_optional)
    spi_quota = 0 if word_target <= 180 else 2 if word_target <= 450 else 4
    if format_plan.get("difficulty") == "hard":
        spi_quota = max(spi_quota, 4)
    optional = spi_optional[:spi_quota] + pii_optional + regex_optional + spi_optional[spi_quota:]
    for name in required + optional:
        if name not in candidates and profile.get(name):
            candidates.append(name)
    selected = required + [name for name in optional if name not in required][: max(0, target - len(required))]
    minimum_selected = min(max(low, LOCKED_FIELDS_MIN), len(candidates))
    selected = selected[: min(len(selected), max(high, minimum_selected))]
    if len(selected) < minimum_selected:
        filler = [name for name in candidates if name not in selected]
        selected.extend(filler[: minimum_selected - len(selected)])
    all_profile_fields = [k for k, v in profile.items() if v and k not in {"profile_id", "source_profile", "birth_year", "occupation", "nationality", "gender", "ethnicity", "religion", "marital_status"}]
    forbidden = [str(profile[k]) for k in all_profile_fields if k not in selected and profile.get(k)]
    return {
        "record_tags": event_background.get("record_tags", []),
        "selected_event_ids": [e.get("event_id") for e in event_background.get("event_list", [])],
        "selected_attributes": selected,
        "estimated_word_target": word_target,
        "target_unique_legal_attributes": {"min": low, "max": high, "selected": len(selected)},
        "locked_values_draft": [str(profile[n]) for n in selected],
        "retention_context": ["record_type", "agency", "channel"],
        "forbidden_values": forbidden,
        "form_matched_fields": form_blank.get("matched_fields", []),
    }

def validate_content_plan(content_plan, profile, form_blank):
    reasons = []
    selected = content_plan.get("selected_attributes") or []
    target_info = content_plan.get("target_unique_legal_attributes") or {}
    target_min = int(target_info.get("min") or 0)
    min_selected = min(max(LOCKED_FIELDS_MIN, target_min), len([name for name in PREFERRED_FIELDS if profile.get(name)]))
    if len(selected) < min_selected:
        reasons.append("too_few_selected_attributes")
    for name in selected:
        if not profile.get(name):
            reasons.append(f"profile_missing:{name}")
    matched = set(form_blank.get("matched_fields") or [])
    if matched and not (set(selected) & matched):
        reasons.append("no_selected_field_matches_form")
    return not reasons, reasons

def mutate_last_digit(value):
    text = str(value or "")
    for idx in range(len(text) - 1, -1, -1):
        if text[idx].isdigit():
            new_digit = str((int(text[idx]) + 9) % 10)
            return text[:idx] + new_digit + text[idx + 1:]
    return text + "0"

def build_adversarial_controls(profile, content_plan, format_plan, row_index):
    factors = set(format_plan.get("difficulty_factors") or [])
    selected = set(content_plan.get("selected_attributes") or [])
    controls = {"distractors": [], "extra_fields": []}
    seed = f"{row_index}|{profile.get('profile_id')}|{format_plan.get('difficulty')}"

    def add_distractor(name, value, kind, context):
        if value and value not in {item.get("value") for item in controls["distractors"]}:
            controls["distractors"].append({
                "name": name,
                "value": str(value),
                "kind": kind,
                "context": context,
                "annotate": False,
                "source": "difficulty_policy",
            })

    if factors.intersection({"hard_adversarial_numbers", "non_pii_codes_near_ids", "multiple_numeric_distractors"}):
        add_distractor("receipt_code_12d", stable_digits(seed + "|receipt", 12), "admin_receipt_code", "mã biên nhận/mã hồ sơ nghiệp vụ, không phải CCCD/CMND cá nhân")
        add_distractor("procedure_code", "HS-" + stable_digits(seed + "|procedure", 6), "admin_procedure_code", "mã hồ sơ thủ tục, không phải định danh cá nhân")
    if factors.intersection({"non_pii_dates_near_dob", "hard_adversarial_numbers"}):
        day = int(stable_digits(seed + "|day", 2)) % 28 + 1
        month = int(stable_digits(seed + "|month", 2)) % 12 + 1
        year = 2024 + (int(stable_digits(seed + "|year", 1)) % 3)
        add_distractor("received_date", f"{day:02d}/{month:02d}/{year}", "admin_received_date", "ngày tiếp nhận/ngày văn bản, không phải ngày sinh")
    if factors.intersection({"agency_person_name_distractor", "multi_person_context"}):
        officer_names = ["Trần Minh Quân", "Phạm Thu Hà", "Lê Hoàng Nam", "Đỗ Minh Châu", "Ngô Hải Yến"]
        add_distractor("officer_name", officer_names[int(stable_digits(seed + "|officer", 2)) % len(officer_names)], "officer_name", "tên cán bộ xử lý/cán bộ tiếp nhận, không phải chủ thể dữ liệu trong profile")

    # Controlled corrected-value cases are PII because the old/wrong value is still tied to the data subject.
    # They are allowed only when the difficulty policy explicitly selects this factor.
    if "corrected_identifier" in factors and "cccd" in selected and profile.get("cccd"):
        wrong_value = mutate_last_digit(profile["cccd"])
        meta = PII_META.get("cccd", {})
        controls["extra_fields"].append({
            "name": "cccd",
            "value": wrong_value,
            "label": str(meta.get("label") or meta.get("sensitivity") or "PII"),
            "sensitivity": str(meta.get("sensitivity") or "PII"),
            "identifier_type": str(meta.get("identifier_type") or "direct"),
            "context_dependency": str(meta.get("context_dependency") or "context_free"),
            "source": "difficulty_policy",
            "annotate": True,
            "level": meta.get("level"),
            "name_vi": meta.get("name_vi"),
            "law_ref": meta.get("law_ref"),
            "value_status": "erroneous_previous",
        })
    return controls

def apply_adversarial_controls(injection, controls):
    injection = {
        "fields": list(injection.get("fields") or []),
        "context_seeds": list(injection.get("context_seeds") or []),
        "locked_values": list(injection.get("locked_values") or []),
    }
    for field in controls.get("extra_fields") or []:
        value = str(field.get("value") or "")
        if value and value not in {str(item.get("value")) for item in injection["fields"]}:
            injection["fields"].append(field)
            injection["locked_values"].append(value)
    return injection

In [54]:
# CELL 5 - PROMPTS AND BACKENDS
PROMPT_B10_GENERATE = """
Bạn là hệ thống sinh dữ liệu huấn luyện NER tiếng Việt cho miền hồ sơ hành chính.

Viết một đoạn văn hoặc hai đoạn văn xuôi tự nhiên, liên quan trực tiếp tới thủ tục/hồ sơ dưới đây. Không viết bảng, bullet, JSON, hoặc key-value cứng.

THỦ TỤC:
- record_type: {record_type}
- agency: {agency}
- field/sub_field: {field} / {sub_field}
- form_name: {form_name}

BỐI CẢNH MẪU:
{background_context}

KẾ HOẠCH VĂN PHONG:
{format_plan_json}

CHỈ DẪN VĂN PHONG CỤ THỂ:
{narrative_directives}

CONTEXT SEEDS - được phép dùng làm ngữ cảnh, KHÔNG cần annotate:
{context_seeds_json}

GIÁ TRỊ GÂY NHIỄU ĐƯỢC PHÉP - chỉ dùng nếu tự nhiên, KHÔNG biến thành thông tin cá nhân:
{adversarial_distractors_block}

GIÁ TRỊ BẮT BUỘC PHẢI XUẤT HIỆN NGUYÊN VẸN:
{locked_values_block}

RÀNG BUỘC CỨNG:
1. Mỗi locked_value phải xuất hiện chính xác trong text cuối.
2. Không bịa thêm CCCD, số điện thoại, email, hộ chiếu, ngày sinh, tài khoản, mã định danh khác ngoài locked_values.
3. Các giá trị gây nhiễu nếu dùng phải nằm trong ngữ cảnh mã hồ sơ/ngày văn bản/cán bộ xử lý, không được mô tả là PII/SPI của người nộp.
4. Không dùng placeholder như [NAME], [PHONE], XXX.
5. Không nhắc NER, không nhắc locked_values.
6. Văn bản phải là tiếng Việt hành chính tự nhiên.
7. Độ dài mục tiêu: từ {target_min_words} đến {target_max_words} từ.
8. Không trả lời kiểu nhận xét, đánh giá, giải thích, "Tuyệt vời", "Dưới đây là"; chỉ viết nội dung hồ sơ.
9. Không mở đầu mọi văn bản bằng tên cơ quan; hãy thay đổi điểm nhìn theo chỉ dẫn văn phong.

Chỉ trả về văn bản cuối cùng.
""".strip()

PROMPT_B10_AUTOFIX = """
Viết lại TOÀN BỘ văn bản dưới đây thành văn xuôi hành chính tự nhiên. Giữ nguyên ngữ cảnh hồ sơ và chèn các giá trị còn thiếu vào câu phù hợp.

GIÁ TRỊ CÒN THIẾU:
{missing_block}

TOÀN BỘ GIÁ TRỊ BẮT BUỘC PHẢI XUẤT HIỆN NGUYÊN VẸN:
{locked_values_block}

VĂN BẢN CŨ:
{old_text}

RÀNG BUỘC:
1. Chỉ trả về văn bản hồ sơ cuối cùng. Không nhận xét, không viết "Tuyệt vời", không giải thích.
2. Không thêm danh sách giá trị ở cuối văn bản.
3. Mỗi giá trị còn thiếu phải nằm trong câu có ngữ cảnh đúng với loại trường.
4. Không viết bảng, bullet, JSON, markdown, key-value.
5. Không bịa thêm số định danh ngoài danh sách bắt buộc.
""".strip()

PROMPT_B10_SHORTEN = """
Rút gọn văn bản dưới đây để nằm trong khoảng {target_min_words}-{target_max_words} từ.

YÊU CẦU BẮT BUỘC:
- Giữ nguyên mọi giá trị bắt buộc sau, không xóa, không sửa chính tả, không đổi định dạng:
{locked_values_block}
- Không thêm CCCD, số điện thoại, email, hộ chiếu, ngày sinh, tài khoản, mã định danh khác ngoài danh sách trên.
- Không dùng tên khóa kỹ thuật như full_name, marital_status, locked_values, context_seeds, injection_list.
- Bỏ câu chung chung, câu lặp ý, phần cam kết dài; giữ thủ tục, cơ quan, vai trò người liên hệ/đại diện và các thông tin cá nhân bắt buộc.
- Không dùng bullet, bảng, JSON, markdown.

VĂN BẢN CẦN RÚT GỌN:
{old_text}

Chỉ trả về văn bản đã rút gọn.
""".strip()


def build_prompt_b10(context):
    info = context["record_type_info"]
    return PROMPT_B10_GENERATE.format(
        record_type=info.get("record_type"), agency=info.get("agency"), field=info.get("field"),
        sub_field=info.get("sub_field"), form_name=info.get("form_name"),
        background_context=context["background_context"],
        format_plan_json=json.dumps(context["format_plan"], ensure_ascii=False),
        narrative_directives=build_narrative_directives(context["format_plan"]),
        context_seeds_json=json.dumps(context["injection_list"].get("context_seeds", []), ensure_ascii=False),
        adversarial_distractors_block=format_adversarial_distractors(context),
        locked_values_block="\n".join(f'- "{v}"' for i, v in enumerate(context["injection_list"]["locked_values"], 1)),
        target_min_words=context["length_plan"]["target_min_words"],
        target_max_words=context["length_plan"]["target_max_words"],
    )

def format_adversarial_distractors(context):
    distractors = (context.get("adversarial_controls") or {}).get("distractors") or []
    if not distractors:
        return "(không có)"
    return "\n".join(f'- "{item.get("value")}": {item.get("context")}' for item in distractors)

def build_narrative_directives(format_plan):
    voice = format_plan.get("voice_detail") or {}
    tone = format_plan.get("tone_detail") or {}
    structure = format_plan.get("structure_detail") or {}
    style = format_plan.get("style_detail") or {}
    tag = format_plan.get("record_tag_detail") or {}
    return "\n".join([
        f"- Dạng văn bản: {tag.get('label') or format_plan.get('record_tag')} - {tag.get('description') or 'bám đúng ngữ cảnh hồ sơ.'}",
        f"- Góc nhìn/người kể: {voice.get('label') or format_plan.get('voice')} - {voice.get('description') or 'giữ nhất quán trong toàn văn.'}",
        f"- Giọng điệu: {tone.get('label') or format_plan.get('tone')} - {tone.get('description') or 'không sáo rỗng.'}",
        f"- Cấu trúc: {structure.get('label') or format_plan.get('structure')} - {structure.get('description') or 'không liệt kê máy móc.'}",
        f"- Phong cách: {style.get('label') or format_plan.get('style')}.",
        "- Có thể mở đầu bằng người nộp, vấn đề hồ sơ, kênh trao đổi, tài liệu kèm theo, hoặc ghi chú rà soát.",
        "- Tránh công thức lặp như 'Nhóm thông tin...' nếu không thật cần; hãy nhúng PII/SPI vào câu văn tự nhiên.",
    ])

FIELD_LABELS = {
    "full_name": "họ và tên",
    "address": "địa chỉ",
    "phone": "số điện thoại",
    "cccd": "so CCCD/CMND",
    "email": "email",
    "dob": "ngày sinh",
    "passport_number": "số hộ chiếu",
    "gender": "giới tính",
    "nationality": "quốc tịch",
    "marital_status": "tình trạng hôn nhân",
    "ethnicity": "dân tộc",
    "religion": "tôn giáo",
    "bank_account": "số tài khoản ngân hàng",
    "driver_license": "số giấy phép lái xe",
    "vehicle_plate": "biển số xe",
    "health_status": "tình trạng sức khỏe",
    "criminal_record": "dữ liệu tội phạm, vi phạm pháp luật",
    "family_relations": "thông tin mối quan hệ gia đình",
    "digital_account": "tài khoản số",
    "behavioral_data": "hành vi sử dụng dịch vụ mạng",
    "political_view": "quan điểm chính trị",
    "private_life": "đời sống riêng tư",
    "biometric": "dữ liệu sinh trắc học",
    "location_data": "dữ liệu vị trí",
    "sexual_orientation": "xu hướng tính dục",
}

def field_label(field):
    name = field.get("name") if isinstance(field, dict) else str(field)
    return FIELD_LABELS.get(name) or (field.get("name_vi") if isinstance(field, dict) else None) or name

def stable_index(seed_text, n):
    if n <= 0:
        return 0
    digest = hashlib.sha256(str(seed_text).encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % n

def stable_pick(items, seed_text, fallback=None):
    items = list(items or [])
    if not items:
        return fallback
    return items[stable_index(seed_text, len(items))]

FIELD_GROUPS = {
    "identity": {"full_name", "dob", "gender", "nationality", "ethnicity", "marital_status", "family_relations"},
    "contact": {"address", "phone", "email", "digital_account", "location_data"},
    "documents": {"cccd", "passport_number", "driver_license", "vehicle_plate", "bank_account", "eid_credentials"},
    "sensitive": {"health_status", "criminal_record", "religion", "political_view", "private_life", "behavioral_data", "biometric", "sexual_orientation"},
}

def render_field_clause(field):
    name = field.get("name")
    value = str(field.get("value") or "")
    if field.get("value_status") == "erroneous_previous":
        return f"số CCCD/CMND từng ghi nhầm là {value} trước khi được đính chính"
    grammar = (style_registry.get("narrative_grammar") or {}) if "style_registry" in globals() else {}
    realizers = (grammar.get("field_realizers") or {}).get(name) or []
    template = stable_pick(realizers, f"{name}|{value}")
    if template and "?" not in str(template):
        return str(template).format(value=value)
    if name == "full_name":
        return f"họ và tên {value}"
    if name == "dob":
        return f"ngày sinh {value}"
    if name == "address":
        return f"địa chỉ liên hệ {value}"
    if name == "phone":
        return f"số điện thoại {value}"
    if name == "email":
        return f"email {value}"
    if name == "cccd":
        return f"số CCCD/CMND {value}"
    if name == "passport_number":
        return f"số hộ chiếu {value}"
    if name == "driver_license":
        return f"số giấy phép lái xe {value}"
    if name == "vehicle_plate":
        return f"biển số xe {value}"
    if name == "bank_account":
        return f"số tài khoản ngân hàng {value}"
    if name == "digital_account":
        return f"tài khoản số {value}"
    if name == "ethnicity":
        return f"dân tộc {value}"
    if name == "marital_status":
        return f"tình trạng hôn nhân {value}"
    if name == "religion":
        return f"tôn giáo {value}"
    if name == "health_status":
        return f"tình trạng sức khỏe {value}"
    if name == "criminal_record":
        return f"dữ liệu tội phạm, vi phạm pháp luật {value}"
    if name == "family_relations":
        return f"thông tin mối quan hệ gia đình {value}"
    if name == "behavioral_data":
        return f"hành vi sử dụng dịch vụ mạng {value}"
    if name == "political_view":
        return f"quan điểm chính trị {value}"
    if name == "private_life":
        return f"đời sống riêng tư {value}"
    if name == "biometric":
        return f"dữ liệu sinh trắc học {value}"
    if name == "location_data":
        return f"dữ liệu vị trí {value}"
    if name == "sexual_orientation":
        return f"xu hướng tính dục {value}"
    return f"{field_label(field)} {value}"

def render_adversarial_distractor_clause(item):
    value = str(item.get("value") or "")
    kind = item.get("kind")
    if not value:
        return ""
    if kind == "admin_received_date":
        return f"ngày tiếp nhận văn bản là {value}"
    if kind == "admin_receipt_code":
        return f"mã biên nhận nghiệp vụ là {value}, không phải số định danh cá nhân"
    if kind == "admin_procedure_code":
        return f"mã hồ sơ thủ tục là {value}"
    if kind == "officer_name":
        return f"cán bộ xử lý được ghi nhận là {value}"
    return f"mã nghiệp vụ tham chiếu là {value}"

def grouped_field_clauses(fields):
    groups = {key: [] for key in FIELD_GROUPS}
    groups["other"] = []
    for field in fields:
        if not field.get("annotate", True) or not field.get("value"):
            continue
        name = field.get("name")
        if name == "full_name":
            continue
        target = "other"
        for group_name, names in FIELD_GROUPS.items():
            if name in names:
                target = group_name
                break
        groups[target].append(render_field_clause(field))
    return groups

def fields_for_values(context, values):
    fields = context.get("injection_list", {}).get("fields", [])
    out = []
    for value in values:
        match = next((f for f in fields if str(f.get("value")) == str(value)), None)
        out.append(match or {"name": "unknown", "value": value})
    return out

def format_missing_fields(context, missing_values):
    return "\n".join(f"- {field_label(f)} ({f.get('name')}): {f.get('value')}" for f in fields_for_values(context, missing_values))

def build_prompt_autofix(old_text, context, missing_values):
    locked_values = context["injection_list"]["locked_values"]
    return PROMPT_B10_AUTOFIX.format(
        missing_block=format_missing_fields(context, missing_values),
        locked_values_block="\n".join(f'- "{v}"' for i, v in enumerate(locked_values, 1)),
        old_text=old_text,
    )

def build_prompt_shorten(context, old_text):
    return PROMPT_B10_SHORTEN.format(
        target_min_words=context["length_plan"]["target_min_words"],
        target_max_words=context["length_plan"]["target_max_words"],
        locked_values_block="\n".join(f'- "{v}"' for i, v in enumerate(context["injection_list"]["locked_values"], 1)),
        old_text=old_text,
    )


def clean_model_output(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:text|json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    return re.sub(r"^(Văn bản cuối cùng|Output|Kết quả)\s*:\s*", "", text, flags=re.I).strip()

class MockBackend:
    def generate(self, prompt, context, **kwargs):
        locked = context["injection_list"]["locked_values"]
        info = context["record_type_info"]
        p = context["profile_fake"]
        min_words = (context.get("length_plan") or {}).get("target_min_words", 120)
        text = f"Trong quá trình chuẩn bị {info.get('record_type')}, cán bộ tại {info.get('agency')} ghi nhận hồ sơ của {locked[0]}. Người này hiện là {p.get('occupation')} và cung cấp các thông tin đối chiếu gồm {', '.join(locked[1:])}. Nội dung được trình bày bằng văn xuôi để phục vụ xử lý hồ sơ hành chính, không dùng bảng hay biểu mẫu cứng."
        filler = " Cán bộ tiếp nhận đối chiếu thông tin với biểu mẫu, ghi nhận lý do bổ sung, kiểm tra sự nhất quán giữa hồ sơ giấy và dữ liệu điện tử, đồng thời nhắc người nộp theo dõi phản hồi chính thức qua kênh đã đăng ký."
        while len(text.split()) < min_words:
            text += filler
        return text

class HFBackend:
    def __init__(self, model_name, load_in_4bit=True, gpu_id=0):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        self.torch = torch
        token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
        if not token:
            raise RuntimeError("HF_TOKEN is required for gated HuggingFace models. Add it in Kaggle Secrets.")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, token=token)
        kwargs = {"trust_remote_code": True, "device_map": {"": gpu_id}}
        if load_in_4bit:
            kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
        else:
            kwargs["torch_dtype"] = torch.float16
        self.model = AutoModelForCausalLM.from_pretrained(model_name, token=token, **kwargs)
        self.model.eval()
        if hasattr(self.model, "hf_device_map"):
            print("MODEL_DEVICE_MAP", self.model.hf_device_map)

    def generate(self, prompt, context, **kwargs):
        messages = [{"role": "user", "content": prompt}]
        if hasattr(self.tokenizer, "apply_chat_template") and self.tokenizer.chat_template:
            input_ids = self.tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
            attention_mask = None
        else:
            encoded = self.tokenizer(prompt, return_tensors="pt")
            input_ids = encoded["input_ids"]
            attention_mask = encoded.get("attention_mask")
        if hasattr(input_ids, "data") and "input_ids" in input_ids.data:
            attention_mask = input_ids.data.get("attention_mask")
            input_ids = input_ids.data["input_ids"]
        elif isinstance(input_ids, dict) and "input_ids" in input_ids:
            attention_mask = input_ids.get("attention_mask")
            input_ids = input_ids["input_ids"]
        input_ids = input_ids.to(self.model.device)
        if attention_mask is not None:
            attention_mask = attention_mask.to(self.model.device)
        gen_kwargs = {
            "max_new_tokens": kwargs.get("max_new_tokens", MAX_NEW_TOKENS),
            "do_sample": True,
            "temperature": kwargs.get("temperature", TEMPERATURE),
            "top_p": kwargs.get("top_p", TOP_P),
            "pad_token_id": self.tokenizer.eos_token_id,
        }
        if attention_mask is not None:
            gen_kwargs["attention_mask"] = attention_mask
        with self.torch.no_grad():
            out = self.model.generate(input_ids=input_ids, **gen_kwargs)
        return clean_model_output(self.tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True))

import threading as _threading
_backend_local = _threading.local()
_worker_local = _threading.local()  # holds per-thread diversity_pool
_model_load_lock = _threading.Lock()  # serialize concurrent model loads (bitsandbytes is not thread-safe)

def get_backend(gpu_id=None):
    if not hasattr(_backend_local, "backend") or _backend_local.backend is None:
        with _model_load_lock:
            # re-check inside lock: another thread may have released lock after loading its own model
            if not hasattr(_backend_local, "backend") or _backend_local.backend is None:
                _backend_local.backend = MockBackend() if USE_MOCK_BACKEND else HFBackend(MODEL_NAME, LOAD_IN_4BIT, gpu_id=int(gpu_id) if gpu_id is not None else 0)
    return _backend_local.backend
print("Backend ready (thread-local, lazy init)")

Backend ready (thread-local, lazy init)


In [55]:
# CELL 6 - VALIDATION, ANNOTATION, CHECKPOINT
@dataclass
class ValidationResult:
    ok: bool
    hard_fail: bool = False
    missing_values: list = field(default_factory=list)
    suspicious_values: list = field(default_factory=list)
    reasons: list = field(default_factory=list)

def count_words(text):
    return len(re.findall(r"\w+", str(text), flags=re.UNICODE))

def build_length_plan(form_blank, locked_count):
    form_words = count_words(form_blank.get("raw_without_placeholder") or "")
    if form_words > 0:
        target_min = math.ceil(form_words * FORM_WORD_MIN_RATIO)
        target_max = math.ceil(form_words * FORM_WORD_MAX_RATIO)
    else:
        target_min = max(80, locked_count * MIN_WORDS_PER_LOCKED_VALUE)
        target_max = max(120, target_min)
    target_min = max(target_min, locked_count * MIN_WORDS_PER_LOCKED_VALUE)
    target_min = min(target_min, TARGET_MAX_WORDS)
    target_max = min(max(target_max, target_min), TARGET_MAX_WORDS)
    return {
        "form_words": form_words,
        "locked_count": locked_count,
        "target_min_words": target_min,
        "target_max_words": target_max,
    }

_DATE_SLASH_RE = re.compile(r"^(\d{1,2})/(\d{1,2})/(\d{4})$")
_VIETNAMESE_NAME_RE = re.compile(r"[A-ZÀ-ỴĐ][a-zà-ỵđ]+(?:\s+[A-ZÀ-ỴĐ][a-zà-ỵđ]+){1,3}")
_TECHNICAL_FIELD_KEYS = {"full_name", "marital_status", "passport_number", "bank_account", "driver_license", "vehicle_plate", "health_status", "locked_values", "context_seeds", "injection_list"}
_CASE_INSENSITIVE_FIELDS = {
    "marital_status", "ethnicity", "religion",
    "health_status", "criminal_record", "biometric", "political_view",
    "private_life", "sexual_orientation", "behavioral_data",
    "location_data", "family_relations", "gender", "nationality",
}
_MOJIBAKE_RE = re.compile(r"(?:\?{2,}|�|[A-Za-zÀ-ỵĐđ]\?+\w*|\?+[A-Za-zÀ-ỵĐđ])")
_META_RESPONSE_RE = re.compile(r"(?:Tuyệt vời|bản chỉnh sửa|Dưới đây là|điểm nổi bật|đánh giá về|Bạn đã|yêu cầu và hướng dẫn|Ngôn ngữ\s*:|Định dạng\s*:|Thông tin\s*:|Tuyet voi|ban chinh sua|Duoi day la|diem noi bat|danh gia ve|Ban da|yeu cau va huong dan|Ngon ngu\s*:|Dinh dang\s*:|Thong tin\s*:)", re.IGNORECASE)

def _strip_accents(text):
    text = str(text).replace("đ", "d").replace("Đ", "D")
    normalized = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")

def _edit_distance_at_most_one(left, right):
    if abs(len(left) - len(right)) > 1:
        return False
    i = j = edits = 0
    while i < len(left) and j < len(right):
        if left[i] == right[j]:
            i += 1
            j += 1
            continue
        edits += 1
        if edits > 1:
            return False
        if len(left) == len(right):
            i += 1
            j += 1
        elif len(left) > len(right):
            i += 1
        else:
            j += 1
    return edits + (len(left) - i) + (len(right) - j) <= 1

def find_near_name_variants(text, allowed_full_names=None):
    allowed = [name for name in (allowed_full_names or []) if name]
    if not allowed:
        return []
    allowed_exact = set(allowed)
    allowed_norm = {_strip_accents(name).lower(): name for name in allowed}
    suspicious = []
    for match in _VIETNAMESE_NAME_RE.finditer(str(text)):
        candidate = match.group(0)
        if candidate in allowed_exact:
            continue
        candidate_norm = _strip_accents(candidate).lower()
        for norm_name in allowed_norm:
            if candidate_norm == norm_name:
                suspicious.append(candidate)
                break
            c_parts, n_parts = candidate_norm.split(), norm_name.split()
            if len(c_parts) == len(n_parts) and c_parts[:-1] == n_parts[:-1] and _edit_distance_at_most_one(c_parts[-1], n_parts[-1]):
                suspicious.append(candidate)
                break
    return sorted(set(suspicious))

def _case_insensitive_value_in_text(value, text):
    return re.search(rf"(?<!\w){re.escape(value)}(?!\w)", text, re.IGNORECASE) is not None

def _strip_diacritics(s):
    import unicodedata
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode("ascii")

def _value_in_text(value, text, case_insensitive_values=None):
    if value in text:
        return True
    if value in (case_insensitive_values or set()) and _case_insensitive_value_in_text(value, text):
        return True
    m = _DATE_SLASH_RE.match(value)
    if m:
        d, mo, y = m.group(1), m.group(2), m.group(3)
        vn_forms = [
            f"{int(d)} tháng {int(mo)} năm {y}",
            f"{d} tháng {mo} năm {y}",
            f"{int(d)} tháng {mo} năm {y}",
            f"{d} tháng {int(mo)} năm {y}",
        ]
        if any(f in text for f in vn_forms):
            return True
    # ASCII-only values (emails, usernames, codes): match against diacritic-stripped text
    if value.isascii() and value.strip():
        if value.lower() in _strip_diacritics(text).lower():
            return True
    # Short-to-medium SPI labels (≤10 words): accept if all content words present (paraphrase tolerance)
    _STOP = {"là", "của", "và", "trong", "có", "không", "theo", "về", "tại", "với", "được", "đã", "các", "một", "những"}
    words = value.lower().split()
    content_words = [w for w in words if w not in _STOP] if len(words) > 5 else words
    if 2 <= len(words) <= 10 and content_words:
        text_lower = text.lower()
        if all(w in text_lower for w in content_words):
            return True
    return False

def _technical_field_key_hits(text):
    return sorted(key for key in _TECHNICAL_FIELD_KEYS if re.search(rf"(?<!\w){re.escape(key)}(?!\w)", text))

def find_mojibake_hits(text, limit=8):
    hits = []
    text = str(text)
    for match in _MOJIBAKE_RE.finditer(text):
        start = max(0, match.start() - 12)
        end = min(len(text), match.end() + 12)
        hits.append(text[start:end])
        if len(hits) >= limit:
            break
    return hits

def find_semantic_mismatch_hits(text, limit=8):
    raw = str(text)
    checks = [
        r"(?:cmnd|cccd|căn cước|can cuoc|mã hồ sơ|ma ho so)[^,.\n]{0,40}\bGPLX\d+\b",
        r"\bGPLX\d+\b[^,.\n]{0,40}(?:cmnd|cccd|căn cước|can cuoc|mã hồ sơ|ma ho so)",
        r"(?:mã hồ sơ|ma ho so|cmnd|cccd)[^,.\n]{0,40}\bP\d{6,}\b",
        r"\bP\d{6,}\b[^,.\n]{0,40}(?:mã hồ sơ|ma ho so|cmnd|cccd)",
        r"email[^.\n]{0,50}\b(?![\w.+-]+@[\w.-]+\.[A-Za-z]{2,})[A-Za-z][\w.-]{3,}\.\d{2,}\b",
        r"(?:tính chất kinh tế|tinh chat kinh te|thông tin kinh tế|thong tin kinh te)[^.\n]{0,40}\bKinh\b",
        r"\bKinh\b[^.\n]{0,40}(?:tính chất kinh tế|tinh chat kinh te|thông tin kinh tế|thong tin kinh te)",
    ]
    hits = []
    for pattern in checks:
        for match in re.finditer(pattern, raw, re.IGNORECASE):
            hits.append(match.group(0))
            if len(hits) >= limit:
                return hits
    lowered = raw.lower()
    if "co tien su di ung thuoc" in lowered:
        for match in re.finditer(r"co tien su di ung thuoc[^.\n]{0,100}(?:ung thư|ung thu)|(?:ung thư|ung thu)[^.\n]{0,100}co tien su di ung thuoc", raw, re.IGNORECASE):
            hits.append(match.group(0))
            if len(hits) >= limit:
                return hits
    return hits

def find_allowed_extra_personal_context_hits(text, allowed_extra_values, limit=8):
    raw = str(text)
    personal_kw = re.compile(r"(cccd|cmnd|căn cước|can cuoc|số định danh|so dinh danh|số điện thoại|so dien thoai|email cá nhân|email ca nhan|hộ chiếu|ho chieu|giấy phép lái xe|giay phep lai xe|gplx|tài khoản ngân hàng|tai khoan ngan hang)", re.IGNORECASE)
    admin_kw = re.compile(r"(mã hồ sơ|ma ho so|mã biên nhận|ma bien nhan|mã thủ tục|ma thu tuc|mã nghiệp vụ|ma nghiep vu|số văn bản|so van ban|ngày tiếp nhận|ngay tiep nhan|ngày văn bản|ngay van ban|cán bộ|can bo|không phải|khong phai)", re.IGNORECASE)
    hits = []
    for value in allowed_extra_values or []:
        value = str(value or "")
        if not value or value not in raw:
            continue
        start = 0
        while True:
            pos = raw.find(value, start)
            if pos == -1:
                break
            window = raw[max(0, pos - 70): min(len(raw), pos + len(value) + 70)]
            if personal_kw.search(window) and not admin_kw.search(window):
                hits.append(window)
                if len(hits) >= limit:
                    return hits
            start = pos + len(value)
    return hits

def validate_text(text, locked_values, min_chars=MIN_CHARS, forbidden_values=None, allowed_full_names=None, case_insensitive_values=None, allowed_extra_values=None, min_words=None, max_words=None):
    ci_values = set(case_insensitive_values or [])
    allowed_extra_values = [str(v) for v in (allowed_extra_values or []) if v]
    reasons, missing = [], [v for v in locked_values if v and not _value_in_text(v, text, ci_values)]
    if len(text.strip()) < min_chars:
        reasons.append("text_too_short")
    word_count = count_words(text)
    if min_words is not None and word_count < min_words:
        reasons.append("word_count_too_short")
    if max_words is not None and word_count > max_words:
        reasons.append("word_count_too_long")
    if missing:
        reasons.append("missing_locked_values")
    allowed = set()
    for value in locked_values:
        allowed.update(re.findall(r"\d{6,}", value))
    for value in allowed_extra_values:
        allowed.update(re.findall(r"\d{6,}", value))
    suspicious = [m for m in re.findall(r"\b\d{8,12}\b", text) if m not in allowed]
    if suspicious:
        reasons.append("suspicious_numeric_value")
    allowed_value_set = set(locked_values or []) | set(allowed_extra_values)
    forbidden_hits = [v for v in (forbidden_values or []) if v and len(v) >= 4 and v in text and v not in allowed_value_set]
    if forbidden_hits:
        reasons.append("forbidden_value_present")
    near_name_variants = find_near_name_variants(text, allowed_full_names)
    if near_name_variants:
        reasons.append("near_name_variant_present")
    technical_hits = _technical_field_key_hits(text)
    if technical_hits:
        reasons.append("technical_field_key_present")
    if re.search(r"(?m)^\s*(?:[*+-]\s+|\d+[.)]\s+|#{1,6}\s+|\*\*[^*\n]{3,}\*\*\s*$)", str(text)):
        reasons.append("non_prose_markdown_or_bullet")
    placeholder_hits = re.findall(r"\[[^\]]+\]|X{2,}", str(text))
    if placeholder_hits:
        reasons.append("placeholder_present")
    mojibake_hits = find_mojibake_hits(text)
    if mojibake_hits:
        reasons.append("mojibake_or_garbled_text")
    meta_response = bool(_META_RESPONSE_RE.search(str(text))) and len(missing) >= max(1, len(locked_values) // 2)
    if meta_response:
        reasons.append("meta_response_or_instruction_followup")
    semantic_hits = find_semantic_mismatch_hits(text)
    if semantic_hits:
        reasons.append("semantic_field_value_mismatch")
    extra_context_hits = find_allowed_extra_personal_context_hits(text, allowed_extra_values)
    if extra_context_hits:
        reasons.append("adversarial_distractor_used_as_personal_identifier")
    hard = bool(missing or suspicious or forbidden_hits or near_name_variants or technical_hits or placeholder_hits or mojibake_hits or meta_response or semantic_hits or extra_context_hits or "word_count_too_long" in reasons or "non_prose_markdown_or_bullet" in reasons)
    return ValidationResult(not reasons, hard, missing, suspicious + forbidden_hits + near_name_variants + technical_hits + placeholder_hits + mojibake_hits + semantic_hits + extra_context_hits, reasons)

def apply_optional_noise(text, locked_values, rng):
    if rng.random() >= NOISE_PROBABILITY:
        return text
    candidate = text.replace(" hồ sơ ", " hồ sơ  ", 1)
    return text if validate_text(candidate, locked_values).hard_fail else candidate

def _date_patterns(value):
    m = _DATE_SLASH_RE.match(str(value))
    if not m:
        return []
    d, mo, y = m.groups()
    day_re = f"(?:{re.escape(d)}|{re.escape(str(int(d)))})"
    month_re = f"(?:{re.escape(mo)}|{re.escape(str(int(mo)))})"
    year_re = re.escape(y)
    return [
        re.compile(rf"(?<!\d){day_re}\s*/\s*{month_re}\s*/\s*{year_re}(?!\d)"),
        re.compile(rf"(?<!\d){day_re}\s*-\s*{month_re}\s*-\s*{year_re}(?!\d)"),
        re.compile(rf"(?<!\d){day_re}\s*\.\s*{month_re}\s*\.\s*{year_re}(?!\d)"),
        re.compile(rf"(?<!\d){day_re}\s+th(?:á|a|Ã¡)ng\s+{month_re}\s+n(?:ă|a|Äƒ)m\s+{year_re}(?!\d)", re.IGNORECASE),
    ]

def _iter_field_matches(text, field):
    value = str(field.get("value") or "")
    if field.get("name") == "dob":
        seen = set()
        for pattern in _date_patterns(value):
            for match in pattern.finditer(text):
                span = (match.start(), match.end())
                seen.add(span)
                yield span
        start = 0
        while value:
            pos = text.find(value, start)
            if pos == -1:
                break
            span = (pos, pos + len(value))
            if span not in seen:
                yield span
            start = pos + len(value)
        return
    if field.get("name") not in REGEX_LIKE_FIELDS:
        # Exact case-sensitive first (mirrors _value_in_text); fallback to IGNORECASE
        _start, _found = 0, False
        while True:
            _pos = text.find(value, _start)
            if _pos == -1:
                break
            _found = True
            yield _pos, _pos + len(value)
            _start = _pos + len(value)
        if not _found:
            for _m in re.finditer(re.escape(value), text, re.IGNORECASE):
                yield _m.start(), _m.end()
        return
    start = 0
    while True:
        pos = text.find(value, start)
        if pos == -1:
            break
        yield pos, pos + len(value)
        start = pos + len(value)

def annotate_exact_spans(text, fields):
    spans, seen = [], set()
    for field in fields:
        if not field.get("annotate", True) or not field.get("value"):
            continue
        for pos, end in _iter_field_matches(text, field):
            key = (pos, end, field.get("name"))
            if key not in seen:
                spans.append({"start": pos, "end": end, "text": text[pos:end], "label": field.get("label") or field.get("sensitivity") or "PII", "field_name": field.get("name"), "law_ref": field.get("law_ref")})
                seen.add(key)
    return sorted(spans, key=lambda s: (s["start"], s["end"], s["field_name"] or ""))

def validate_spans(text, spans, fields=None):
    errors = [f"span_mismatch:{s.get('field_name')}:{s.get('start')}:{s.get('end')}" for s in spans if text[s["start"]:s["end"]] != s["text"]]
    if fields is not None:
        present = {s.get("field_name") for s in spans}
        for field in fields:
            if field.get("annotate", True) and field.get("value") and field.get("name") not in present:
                errors.append(f"missing_span:{field.get('name')}")
    return errors

def density_metrics(spans, fields, text=""):
    field_by_name = {f.get("name"): f for f in fields}
    span_field_counts = Counter(s.get("field_name") for s in spans)
    unique_names = {name for name in span_field_counts if name}
    pii_count = spi_count = regex_like_count = context_dependent_count = 0
    for name in unique_names:
        field = field_by_name.get(name) or {}
        if str(field.get("sensitivity") or "").upper() == "SPI":
            spi_count += 1
        else:
            pii_count += 1
        if name in REGEX_LIKE_FIELDS:
            regex_like_count += 1
        if field.get("context_dependency") == "context_dependent":
            context_dependent_count += 1
    span_count = len(spans)
    full_name_spans = span_field_counts.get("full_name", 0)
    return {
        "unique_legal_attribute_count": len(unique_names),
        "span_count": span_count,
        "word_count": len(re.findall(r"\w+", str(text or ""), flags=re.UNICODE)),
        "pii_attribute_count": pii_count,
        "spi_attribute_count": spi_count,
        "regex_like_attribute_count": regex_like_count,
        "context_dependent_attribute_count": context_dependent_count,
        "full_name_span_count": full_name_spans,
        "full_name_span_ratio": round(full_name_spans / span_count, 4) if span_count else 0.0,
        "span_field_counts": dict(span_field_counts),
    }

def density_quality_reasons(metrics, hard_eval=False):
    reasons = []
    min_unique = 6
    word_count = int(metrics.get("word_count") or 0)
    min_spans = max(6, min(10, word_count // 40))
    unique_count = int(metrics.get("unique_legal_attribute_count") or 0)
    span_count = int(metrics.get("span_count") or 0)
    if unique_count < min_unique:
        reasons.append(f"too_few_unique_legal_attributes:{unique_count}")
    if span_count < min_spans:
        reasons.append(f"too_few_spans:{span_count}")
    return reasons

def density_warning_reasons(metrics, hard_eval=False):
    warnings = []
    unique_count = int(metrics.get("unique_legal_attribute_count") or 0)
    span_count = int(metrics.get("span_count") or 0)
    if unique_count < 8:
        warnings.append(f"below_soft_unique_legal_attributes:{unique_count}")
    if span_count < 15:
        warnings.append(f"below_soft_spans:{span_count}")
    if float(metrics.get("full_name_span_ratio") or 0.0) > 0.30:
        warnings.append("full_name_span_ratio_gt_30pct")
    if unique_count and float(metrics.get("regex_like_attribute_count") or 0) / unique_count > 0.40:
        warnings.append("regex_like_attribute_ratio_gt_40pct")
    if hard_eval:
        if unique_count < 18:
            warnings.append(f"below_hard_eval_target_unique_attrs:{unique_count}")
        if span_count < 35:
            warnings.append(f"below_hard_eval_target_spans:{span_count}")
    return warnings

def compact_overlong_text(context):
    info = context.get("record_type_info") or {}
    profile = context.get("profile_fake") or {}
    injection = context.get("injection_list") or {}
    length_plan = context.get("length_plan") or {}
    event = (context.get("event_list") or [{}])[0]
    agency = info.get("agency") or event.get("agency_context") or "cơ quan tiếp nhận"
    record_type = str(info.get("record_type") or "thủ tục hành chính").strip().rstrip(".")
    channel = event.get("channel") or "kênh tiếp nhận hồ sơ"
    full_name = profile.get("full_name") or "người liên hệ"
    occupation = profile.get("occupation")
    birth_year = profile.get("birth_year")

    grouped = grouped_field_clauses(injection.get("fields", []))

    role_bits = [str(full_name)]
    if occupation:
        role_bits.append(str(occupation))
    if birth_year:
        role_bits.append(f"sinh năm {birth_year}")
    role = ", ".join(role_bits)

    opener_variants = [
        [
            f"Người liên hệ hoặc đại diện trong hồ sơ là {role}.",
            f"Hồ sơ liên quan đến thủ tục {record_type} được chuyển qua {channel} để {agency} xử lý.",
        ],
        [
            f"Trong phần bổ sung cho thủ tục {record_type}, thông tin của {role} được đưa vào hồ sơ để đối chiếu.",
            f"Kênh tiếp nhận là {channel}, cơ quan xử lý là {agency}.",
        ],
        [
            f"Hồ sơ {record_type} cần làm rõ một số thông tin cá nhân của {role}.",
            f"Nội dung được ghi nhận qua {channel} trong quá trình xử lý tại {agency}.",
        ],
        [
            f"Qua {channel}, {role} cung cấp thêm thông tin cho thủ tục {record_type}.",
            f"Các nội dung này được dùng trong bước rà soát của {agency}.",
        ],
        [
            f"Khi rà soát tài liệu kèm theo thủ tục {record_type}, hồ sơ ghi nhận thông tin của {role}.",
            f"Phần trao đổi được thực hiện qua {channel} và chuyển tới {agency}.",
        ],
        [
            f"{role} là người được nêu trong phần thông tin cần đối chiếu của hồ sơ.",
            f"Nội dung này gắn với thủ tục {record_type}, phát sinh qua {channel}.",
        ],
        [
            f"Ở bước kiểm tra biểu mẫu, cán bộ ghi nhận hồ sơ của {role} còn một số dữ liệu cần xác nhận.",
            f"Thủ tục liên quan là {record_type}, cơ quan xử lý là {agency}.",
        ],
        [
            f"Trong email hoặc trao đổi bổ sung, {role} xác nhận lại thông tin phục vụ thủ tục {record_type}.",
            f"Các dữ liệu được lưu cùng hồ sơ để {agency} tiếp tục xử lý.",
        ],
        [
            f"Biểu mẫu của thủ tục {record_type} có phần thông tin cá nhân liên quan đến {role}.",
            f"Hồ sơ được cập nhật qua {channel} trước khi chuyển sang bước thẩm định.",
        ],
        [
            f"Phần giải trình bổ sung nêu rõ trường hợp của {role}.",
            f"Nội dung giải trình phục vụ việc đối chiếu hồ sơ {record_type} tại {agency}.",
        ],
        [
            f"Từ tài liệu người nộp gửi qua {channel}, hệ thống ghi nhận một số thông tin của {role}.",
            f"Các thông tin này được gắn với thủ tục {record_type}.",
        ],
        [
            f"Hồ sơ cần bổ sung cho {record_type} không chỉ nêu tài liệu chuyên môn mà còn có dữ liệu cá nhân của {role}.",
            f"Cơ quan xử lý được ghi nhận là {agency}.",
        ],
        [
            f"Trong quá trình chuẩn bị hồ sơ {record_type}, {role} được xác định là đầu mối liên hệ.",
            f"Việc trao đổi diễn ra qua {channel}.",
        ],
        [
            f"Bản ghi xử lý nội bộ nhắc đến {role} trong hồ sơ {record_type}.",
            f"Các trường thông tin được dùng để đối chiếu trước khi {agency} hoàn tất kiểm tra.",
        ],
    ]
    grammar = (style_registry.get("narrative_grammar") or {}) if "style_registry" in globals() else {}
    opening_grammar = grammar.get("opening_grammar") or {}
    if opening_grammar:
        seed = f"{info.get('record_index')}|{record_type}|{full_name}|{channel}"
        actor = stable_pick(opening_grammar.get("actor_phrases"), seed + "|actor", "Người liên hệ trong hồ sơ")
        action = stable_pick(opening_grammar.get("action_phrases"), seed + "|action", "được dùng để đối chiếu thông tin cho")
        procedure = stable_pick(opening_grammar.get("procedure_phrases"), seed + "|procedure", "thủ tục {record_type}").format(record_type=record_type)
        context_phrase = stable_pick(opening_grammar.get("context_phrases"), seed + "|context", "qua {channel}").format(channel=channel)
        sentences = [
            f"{actor} {role} {action} {procedure} {context_phrase}.",
            f"Cơ quan xử lý được ghi nhận là {agency}, nhưng nội dung chính tập trung vào dữ liệu cần đối chiếu trong hồ sơ.",
        ]
    else:
        sentences = list(opener_variants[int(info.get("record_index") or 0) % len(opener_variants)])
    distractor_clauses = [
        clause for clause in (
            render_adversarial_distractor_clause(item)
            for item in ((context.get("adversarial_controls") or {}).get("distractors") or [])
        )
        if clause
    ]
    if distractor_clauses:
        sentences.append("Để tránh nhầm với thông tin cá nhân, hồ sơ cũng ghi riêng " + "; ".join(distractor_clauses) + ".")
    group_titles = [
        {
            "identity": "Các dữ liệu định danh và quan hệ cá nhân được ghi nhận gồm ",
            "contact": "Phần liên hệ và tài khoản dùng để đối chiếu có ",
            "documents": "Ở nhóm giấy tờ, tài chính và mã số liên quan có ",
            "sensitive": "Các thông tin nhạy cảm trong phần khai báo gồm ",
            "other": "Một số thông tin khác được ghi nhận là ",
        },
        {
            "identity": "Về nhân thân, hồ sơ nêu ",
            "contact": "Đối với kênh liên hệ, hồ sơ sử dụng ",
            "documents": "Tài liệu và mã số đi kèm thể hiện ",
            "sensitive": "Phần dữ liệu cần bảo vệ ở mức cao hơn có ",
            "other": "Ngoài ra, hồ sơ còn có ",
        },
        {
            "identity": "Khi đối chiếu người liên quan, cán bộ ghi nhận ",
            "contact": "Các đầu mối liên lạc và tài khoản gồm ",
            "documents": "Các giấy tờ hoặc mã tham chiếu gồm ",
            "sensitive": "Ở phần khai báo nhạy cảm, nội dung thể hiện ",
            "other": "Các trường còn lại thể hiện ",
        },
        {
            "identity": "Thông tin nền về cá nhân gồm ",
            "contact": "Thông tin phục vụ liên hệ gồm ",
            "documents": "Thông tin phục vụ xác thực giấy tờ gồm ",
            "sensitive": "Thông tin thuộc nhóm cần hạn chế truy cập gồm ",
            "other": "Thông tin bổ sung gồm ",
        },
    ][int(info.get("record_index") or 0) % 4]
    registry_group_titles = grammar.get("group_label_styles") or []
    if registry_group_titles:
        group_titles = stable_pick(registry_group_titles, f"{info.get('record_index')}|groups", group_titles)
    group_orders = [
        ["identity", "contact", "documents", "sensitive", "other"],
        ["contact", "identity", "sensitive", "documents", "other"],
        ["documents", "identity", "contact", "sensitive", "other"],
        ["sensitive", "identity", "contact", "documents", "other"],
        ["identity", "documents", "sensitive", "contact", "other"],
        ["contact", "documents", "identity", "sensitive", "other"],
    ]
    registry_group_orders = grammar.get("section_order_patterns") or []
    if registry_group_orders:
        group_order = stable_pick(registry_group_orders, f"{info.get('record_index')}|order", group_orders[0])
    else:
        group_order = group_orders[int(info.get("record_index") or 0) % len(group_orders)]
    transition_phrases = grammar.get("transition_phrases") or []
    for idx, group_name in enumerate(group_order):
        if grouped[group_name]:
            prefix = ""
            if transition_phrases and idx > 0:
                prefix = stable_pick(transition_phrases, f"{info.get('record_index')}|{group_name}|transition", "") + " "
            sentences.append(prefix + group_titles[group_name] + ", ".join(grouped[group_name]) + ".")
    closings = grammar.get("closing_patterns") or []
    sentences.append(stable_pick(closings, f"{info.get('record_index')}|closing", "Cán bộ xử lý dùng các thông tin này để kiểm tra biểu mẫu, xác nhận hồ sơ kèm theo và liên hệ khi cần bổ sung tài liệu."))
    text = " ".join(sentences)
    max_words = int(length_plan.get("target_max_words") or 10**9)
    min_words = int(length_plan.get("target_min_words") or 0)
    fillers = [
        " Hồ sơ được ghi nhận để đối chiếu nội bộ.",
        " Cán bộ tiếp nhận ghi chú việc bổ sung.",
        " Nội dung này phục vụ kiểm tra hồ sơ.",
        f" Việc xử lý thực hiện tại {agency}.",
        " Các thông tin nêu trên được giữ theo đúng tài liệu người nộp đã cung cấp.",
        " Bộ phận xử lý chỉ dùng dữ liệu này cho việc rà soát biểu mẫu liên quan.",
        " Nếu cần làm rõ, cơ quan tiếp nhận sẽ liên hệ qua thông tin đã ghi trong hồ sơ.",
        " Kết quả đối chiếu được lưu cùng hồ sơ để phục vụ bước thẩm định tiếp theo.",
        " Nội dung ghi nhận không bổ sung thêm mã định danh ngoài các giá trị đã có.",
        " Người phụ trách hồ sơ kiểm tra lại sự thống nhất giữa bản khai và tài liệu kèm theo.",
        " Việc bổ sung được thực hiện trong phạm vi thủ tục đang xử lý.",
        " Các trường thông tin cá nhân được trình bày trong cùng một mạch tường thuật.",
    ]
    for filler in fillers:
        if count_words(text) >= min_words:
            break
        candidate = text + filler
        if count_words(candidate) <= max_words:
            text = candidate
    return text

def repair_missing_locked_values(text, context, missing_values):
    if not missing_values:
        return text
    clauses = []
    for field in fields_for_values(context, missing_values):
        clauses.append(render_field_clause(field))
    suffix = " Trong phần đối chiếu bổ sung của hồ sơ, " + "; ".join(clauses) + "."
    return (text or "").strip() + suffix

def ensure_min_words(text, context):
    length_plan = context.get("length_plan") or {}
    min_words = int(length_plan.get("target_min_words") or 0)
    max_words = int(length_plan.get("target_max_words") or TARGET_MAX_WORDS)
    fillers = [
        " Hồ sơ được bộ phận tiếp nhận rà soát theo từng mục thông tin đã cung cấp.",
        " Nội dung đối chiếu chỉ phục vụ thủ tục đang xử lý và không thêm mã định danh mới.",
        " Cán bộ phụ trách ghi nhận kết quả kiểm tra trong phiếu xử lý nội bộ.",
        " Nếu cần làm rõ, cơ quan tiếp nhận sẽ liên hệ theo thông tin đã có trong hồ sơ.",
        " Bộ phận xử lý chỉ dùng dữ liệu này cho việc rà soát biểu mẫu liên quan.",
        " Kết quả đối chiếu được lưu cùng hồ sơ để phục vụ bước thẩm định tiếp theo.",
        " Nội dung ghi nhận không bổ sung thêm mã định danh ngoài các giá trị đã có.",
        " Người phụ trách hồ sơ kiểm tra lại sự thống nhất giữa bản khai và tài liệu kèm theo.",
        " Việc bổ sung được thực hiện trong phạm vi thủ tục đang xử lý.",
        " Các trường thông tin cá nhân được trình bày trong cùng một mạch tường thuật.",
        " Thông tin được lưu trữ theo quy định bảo mật nội bộ của cơ quan.",
        " Cán bộ tiếp nhận xác nhận tính đầy đủ của hồ sơ trước khi chuyển bước.",
    ]
    out = str(text or "").strip()
    for filler in fillers:
        if count_words(out) >= min_words:
            break
        candidate = (out + filler).strip()
        if count_words(candidate) <= max_words:
            out = candidate
    return out

def repair_semantic_field_context(text, context):
    profile = context.get("profile_fake") or {}
    FIELD_KEYWORD_MAP = {
        "ethnicity":         ("dân tộc",              ["giới tính", "tôn giáo", "nghề nghiệp"]),
        "religion":          ("tôn giáo",             ["dân tộc", "giới tính", "nghề nghiệp"]),
        "health_status":     ("sức khỏe",             ["dân tộc", "tôn giáo", "giới tính"]),
        "criminal_record":   ("án tích",              ["sức khỏe", "tôn giáo", "dân tộc", "giới tính"]),
        "political_view":    ("quan điểm chính trị",  ["tôn giáo", "dân tộc", "sức khỏe"]),
        "sexual_orientation":("xu hướng tính dục",   ["dân tộc", "tôn giáo", "giới tính"]),
    }
    for field, (correct_kw, wrong_kws) in FIELD_KEYWORD_MAP.items():
        value = profile.get(field) or ""
        if len(value) < 2:
            continue
        pos = 0
        while True:
            idx = text.find(value, pos)
            if idx == -1:
                break
            pre = text[max(0, idx - 55):idx]
            for wrong_kw in wrong_kws:
                kw_pos = pre.rfind(wrong_kw)
                if kw_pos >= 0:
                    abs_kw = max(0, idx - 55) + kw_pos
                    text = text[:abs_kw] + correct_kw + text[abs_kw + len(wrong_kw):]
                    break
            new_idx = text.find(value, max(0, idx - 5))
            pos = (new_idx + len(value)) if new_idx >= 0 else len(text)
    return text

def rule_repair_text(text, context, validation):
    # Only structural repair here; missing values are injected by pre-annotation guarantee
    _REPAIRABLE = {"word_count_too_short", "missing_locked_values"}
    reasons = set(validation.reasons if validation else [])
    missing_count = len((validation.missing_values or []) if validation else [])
    if reasons - _REPAIRABLE or missing_count > 2:
        base = compact_overlong_text(context)
        return ensure_min_words(base, context)
    return ensure_min_words(text, context)

def acceptable_after_rule_repair(validation):
    if validation is None:
        return False
    blocking = set(validation.reasons) - {"word_count_too_short", "semantic_field_value_mismatch", "missing_locked_values", "word_count_too_long", "mojibake_or_garbled_text"}
    return not blocking

def _acceptable_compact_fallback(validation):
    """Lenient check for deterministic compact text: allow missing_locked_values/word_count_too_short."""
    if validation is None:
        return False
    _COMPACT_SOFT = {"word_count_too_short", "missing_locked_values", "semantic_field_value_mismatch"}
    blocking = set(validation.reasons) - _COMPACT_SOFT
    return not blocking

def should_use_deterministic_rebuild(validation, locked_values):
    if validation is None:
        return False
    reasons = set(validation.reasons)
    missing_count = len(validation.missing_values or [])
    missing_most_locked = missing_count >= max(3, int(len(locked_values) * 0.60))
    return (
        "meta_response_or_instruction_followup" in reasons
        or "semantic_field_value_mismatch" in reasons
        or "non_prose_markdown_or_bullet" in reasons
        or "placeholder_present" in reasons
        or ("missing_locked_values" in reasons and missing_most_locked)
    )


def jaccard_similarity(a, b):
    ta = set(re.findall(r"\w+", str(a).lower(), flags=re.UNICODE))
    tb = set(re.findall(r"\w+", str(b).lower(), flags=re.UNICODE))
    return 0.0 if not ta or not tb else len(ta & tb) / len(ta | tb)

class DiversityPool:
    def __init__(self, max_size=100, threshold=0.85):
        self.max_size = max_size
        self.threshold = threshold
        self.texts = []

    def check(self, text):
        max_sim = max((jaccard_similarity(text, old) for old in self.texts), default=0.0)
        return max_sim <= self.threshold, max_sim

    def add(self, text):
        self.texts.append(text)
        if len(self.texts) > self.max_size:
            self.texts = self.texts[-self.max_size:]

def build_failure(record_id, stage, reason, attempt, context, text="", validation=None, error_trace=None):
    inj = context.get("injection_list") or {}
    return {
        "record_id": record_id, "stage": stage, "reason": reason, "attempt": attempt,
        "record_type_info": context.get("record_type_info"),
        "profile_id": (context.get("profile_fake") or {}).get("profile_id"),
        "field_names": [f.get("name") for f in inj.get("fields", [])],
        "locked_values": inj.get("locked_values", []),
        "missing_values": validation.missing_values if validation else [],
        "suspicious_values": validation.suspicious_values if validation else [],
        "text_preview": text[:700], "prompt_version": context.get("prompt_version"),
        "format_plan": context.get("format_plan"), "difficulty": (context.get("format_plan") or {}).get("difficulty"),
        "error_trace": error_trace, "created_at": utc_now_iso(),
    }

class CheckpointWriter:
    def __init__(self, checkpoint_dir, run_id, shard_name):
        self.checkpoint_dir, self.run_id, self.shard_name = Path(checkpoint_dir), run_id, shard_name
        self.records_path = self.checkpoint_dir / "shards" / f"{shard_name}.jsonl"
        self.records_json_path = self.checkpoint_dir / "exports" / f"records_{shard_name}.json"
        self.failures_path = self.checkpoint_dir / "failures" / f"failures_{shard_name}.jsonl"
        self.worker_path = self.checkpoint_dir / "workers" / f"worker_{shard_name}.json"
        self.progress_path = self.checkpoint_dir / f"progress_{shard_name}.json"  # per-shard: avoids race when 2 workers write concurrently
        for d in ["shards", "exports", "failures", "workers", "reports"]:
            (self.checkpoint_dir / d).mkdir(parents=True, exist_ok=True)
        self.completed_ids, self.failed_ids = set(), set()
        if self.worker_path.exists():
            data = load_json(self.worker_path)
            self.completed_ids, self.failed_ids = set(data.get("completed_ids", [])), set(data.get("failed_ids", []))

    def reset_for_fresh_run(self):
        for path in [self.records_path, self.records_json_path, self.failures_path, self.worker_path]:
            if path.exists():
                path.unlink()
        self.completed_ids, self.failed_ids = set(), set()
        self.save_worker()

    def cleanup_old_range_files(self):
        base = self.shard_name
        patterns = [
            self.checkpoint_dir / "shards" / f"{base}_*.jsonl",
            self.checkpoint_dir / "exports" / f"records_{base}_*.json",
            self.checkpoint_dir / "failures" / f"failures_{base}_*.jsonl",
            self.checkpoint_dir / "workers" / f"worker_{base}_*.json",
            self.checkpoint_dir / "reports" / f"report_{base}_*.json",
        ]
        removed = 0
        for pattern in patterns:
            for path in pattern.parent.glob(pattern.name):
                if path.exists():
                    path.unlink()
                    removed += 1
        if removed:
            print("Removed old timestamped checkpoint files:", removed)

    def is_done(self, record_id):
        return record_id in self.completed_ids or record_id in self.failed_ids

    def write_record(self, record):
        append_jsonl(self.records_path, record)
        self.completed_ids.add(record["record_id"])
        if EXPORT_RECORDS_JSON:
            self.export_records_json()
        self.save_worker()

    def write_failure(self, failure):
        append_jsonl(self.failures_path, failure)
        self.failed_ids.add(failure.get("record_id", "unknown"))
        self.save_worker()

    def save_worker(self):
        payload = {
            "schema_version": "secureprep-kaggle-v2", "run_id": self.run_id, "shard_name": self.shard_name,
            "model_source": "mock" if USE_MOCK_BACKEND else MODEL_NAME,
            "completed_ids": sorted(self.completed_ids), "failed_ids": sorted(self.failed_ids),
            "n_ok": len(self.completed_ids), "n_failed": len(self.failed_ids),
            "record_shards": [str(self.records_path)], "records_json": str(self.records_json_path),
            "failure_log": str(self.failures_path), "saved_at": utc_now_iso(),
        }
        write_json_atomic(self.worker_path, payload)
        write_json_atomic(self.progress_path, {"run_id": self.run_id, "n_ok": len(self.completed_ids), "n_failed": len(self.failed_ids), "updated_at": utc_now_iso()})

    def export_records_json(self):
        records = []
        if self.records_path.exists():
            with self.records_path.open("r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        records.append(json.loads(line))
        write_json_atomic(self.records_json_path, records)

In [56]:
# CELL 7 - GENERATION FUNCTIONS
class PipelineError(Exception):
    def __init__(self, failure):
        super().__init__(failure.get("reason", "pipeline_error"))
        self.failure = failure

def _case_insensitive_locked_values(injection):
    return [
        str(f.get("value"))
        for f in injection.get("fields", [])
        if f.get("annotate", True) and f.get("value") and f.get("name") in _CASE_INSENSITIVE_FIELDS
    ]

def _allowed_adversarial_values(context):
    controls = context.get("adversarial_controls") or {}
    return [str(item.get("value")) for item in controls.get("distractors", []) if item.get("value")]

def _generation_max_new_tokens(context):
    target_max = int((context.get("length_plan") or {}).get("target_max_words") or 350)
    return max(180, min(MAX_NEW_TOKENS, int(target_max * 2.3) + 100))

def _shorten_max_new_tokens(context):
    target_max = int((context.get("length_plan") or {}).get("target_max_words") or 350)
    return max(160, min(MAX_NEW_TOKENS, int(target_max * 1.25) + 30))

def _strip_non_prose_artifacts(text):
    """Strip markdown headers/bullets/placeholders from LLM output, preserving value content."""
    lines = []
    for line in str(text).split("\n"):
        s = line.strip()
        if not s:
            lines.append("")
            continue
        # Bold-only header line: skip entirely
        if re.match(r"^\*\*[^*\n]{2,}\*\*\s*$", s):
            continue
        # Markdown heading: skip entirely
        if re.match(r"^#{1,6}\s+", s):
            continue
        # Bullet line: extract content, strip inline bold
        m = re.match(r"^[*+\-]\s+(.*)", s) or re.match(r"^\d+[.).]\s+(.*)", s)
        if m:
            inner = re.sub(r"\*\*([^*]+)\*\*", r"\1", m.group(1)).strip()
            if inner:
                lines.append(inner)
            continue
        # Regular line: strip inline bold
        lines.append(re.sub(r"\*\*([^*]+)\*\*", r"\1", line))
    cleaned = "\n".join(lines)
    cleaned = re.sub(r"\[[^\]]{2,80}\]", "", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip()
    # Strip trailing JSON/schema block that LLM sometimes appends after prose
    _schema_keys = {"full_name","marital_status","passport_number","bank_account",
                    "driver_license","vehicle_plate","health_status","locked_values",
                    "context_seeds","injection_list"}
    _sk_pattern = re.compile(r'"' + r'(?:' + '|'.join(re.escape(k) for k in sorted(_schema_keys)) + r')"[\s]*:')
    _clines, _cut = cleaned.split("\n"), None
    for _i in range(len(_clines) - 1, max(-1, len(_clines) - 50), -1):
        _ln = _clines[_i].strip()
        if _sk_pattern.search(_ln) or _ln in ("{", "}", "},"):
            _cut = _i
        elif _cut is not None and not _ln:
            break
    if _cut is not None:
        cleaned = "\n".join(_clines[:_cut]).rstrip()
    _inline_re = re.compile(r'\{[^{}]*(?:' + '|'.join(re.escape(k) for k in _schema_keys) + r')[^{}]*\}')
    cleaned = _inline_re.sub("", cleaned).strip()
    return cleaned

def split_range_for_parallel(base_start, base_end, shard_count, shard_index):
    shard_count = max(1, int(shard_count or 1))
    shard_index = int(shard_index or 0)
    if shard_index < 0 or shard_index >= shard_count:
        raise ValueError(f"PARALLEL_SHARD_INDEX must be in [0, {shard_count - 1}], got {shard_index}")
    total = max(0, int(base_end) - int(base_start))
    step = math.ceil(total / shard_count) if total else 0
    start = int(base_start) + shard_index * step
    end = min(int(base_end), start + step)
    return start, end

def build_record_type_info(raw_record, index):
    forms = raw_record.get("attached_forms") or []
    first = forms[0] if forms and isinstance(forms[0], dict) else {}
    return {
        "record_index": index, "template_id": raw_record.get("template_id"), "record_type": raw_record.get("record_type"),
        "field": raw_record.get("field"), "sub_field": raw_record.get("sub_field"), "agency": raw_record.get("agency"),
        "procedure_type": raw_record.get("procedure_type"), "form_name": first.get("form_name"),
        "file_name": first.get("file_name"), "source_url": raw_record.get("source_url"),
    }

def source_mapping_warnings(info):
    warnings = []
    agency = str(info.get("agency") or "")
    sub_field = str(info.get("sub_field") or "")
    match = re.search(r"\(([^()]*(?:Bộ|UBND|Sở)[^()]*)\)", sub_field)
    if match:
        owner = match.group(1).strip()
        if agency and owner and agency not in owner and owner not in agency:
            warnings.append(f"agency_sub_field_owner_mismatch:{agency}|{owner}")
    return warnings

def extract_background_context(raw_record, max_chars=1000):
    forms = raw_record.get("attached_forms") or []
    text = forms[0].get("form_raw_text") if forms and isinstance(forms[0], dict) else ""
    return " ".join(str(text or raw_record.get("procedure_raw_text") or "").split())[:max_chars]

def build_context(row_index, raw_record, rng, run_id):
    form_blank = parse_form_blank(raw_record)
    profile = enrich_profile_by_form(make_profile_fake(profile_core, row_index), form_blank, row_index)
    info = build_record_type_info(raw_record, row_index)
    plan = make_format_plan(style_registry, rng, raw_record)
    event_background = build_event_background(profile, info, form_blank, plan)
    content_plan = build_content_plan(profile, form_blank, event_background, plan, rng)
    plan_ok, plan_reasons = validate_content_plan(content_plan, profile, form_blank)
    if not plan_ok:
        context = {"record_id": f"{run_id}_row_{row_index:06d}", "profile_fake": profile, "record_type_info": info, "form_blank": form_blank, "format_plan": plan, "content_plan": content_plan, "injection_list": {"fields": [], "context_seeds": [], "locked_values": []}, "prompt_version": "b10-vietnamese-v2"}
        raise PipelineError(build_failure(context["record_id"], "B7_control_validation", ";".join(plan_reasons), 1, context))
    controls = build_adversarial_controls(profile, content_plan, plan, row_index)
    injection = build_injection_list(profile, rng, plan["difficulty"], word_target=int(content_plan.get("estimated_word_target") or 350), selected_attributes=content_plan["selected_attributes"])
    injection = apply_adversarial_controls(injection, controls)
    length_plan = build_length_plan(form_blank, len(injection["locked_values"]))
    content_plan["adversarial_distractors"] = controls.get("distractors", [])
    content_plan["controlled_extra_fields"] = controls.get("extra_fields", [])
    content_plan["mode"] = "vietnamese_administrative_narrative"
    content_plan["must_include_locked_values"] = True
    content_plan["must_not_annotate_context_seeds"] = True
    return {
        "record_id": f"{run_id}_row_{row_index:06d}",
        "profile_fake": profile, "record_type_info": info, "form_blank": form_blank,
        "background_context": event_background.get("background_context") or extract_background_context(raw_record),
        "event_list": event_background["event_list"],
        "content_plan": content_plan,
        "format_plan": plan, "length_plan": length_plan, "injection_list": injection, "adversarial_controls": controls, "prompt_version": "b10-vietnamese-v2",
    }

def generate_one(row_index, raw_record, rng, run_id):
    context = build_context(row_index, raw_record, rng, run_id)
    locked, text, last_val = context["injection_list"]["locked_values"], "", None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        prompt = build_prompt_b10(context) if attempt == 1 else build_prompt_autofix(text, context, last_val.missing_values if last_val else locked)
        text = _strip_non_prose_artifacts(get_backend().generate(prompt, context, max_new_tokens=_generation_max_new_tokens(context), temperature=TEMPERATURE, top_p=TOP_P))
        text = repair_semantic_field_context(text, context)
        last_val = validate_text(
            text,
            locked,
            forbidden_values=context["content_plan"].get("forbidden_values", []),
            allowed_full_names=[context["profile_fake"]["full_name"]] if context.get("profile_fake", {}).get("full_name") else [],
            case_insensitive_values=_case_insensitive_locked_values(context["injection_list"]),
            allowed_extra_values=_allowed_adversarial_values(context),
            min_words=context["length_plan"]["target_min_words"],
            max_words=context["length_plan"]["target_max_words"],
        )
        if last_val.ok:
            break
        if should_use_deterministic_rebuild(last_val, locked):
            rebuilt_text = compact_overlong_text(context)
            rebuilt_val = validate_text(
                rebuilt_text,
                locked,
                forbidden_values=context["content_plan"].get("forbidden_values", []),
                allowed_full_names=[context["profile_fake"]["full_name"]] if context.get("profile_fake", {}).get("full_name") else [],
                case_insensitive_values=_case_insensitive_locked_values(context["injection_list"]),
            allowed_extra_values=_allowed_adversarial_values(context),
                min_words=context["length_plan"]["target_min_words"],
                max_words=context["length_plan"]["target_max_words"],
            )
            if rebuilt_val.ok or _acceptable_compact_fallback(rebuilt_val):
                text = rebuilt_text
                last_val = rebuilt_val
                break
            if rebuilt_val.missing_values:
                patched_text = repair_missing_locked_values(rebuilt_text, context, rebuilt_val.missing_values)
                patched_val = validate_text(
                    patched_text,
                    locked,
                    forbidden_values=context["content_plan"].get("forbidden_values", []),
                    allowed_full_names=[context["profile_fake"]["full_name"]] if context.get("profile_fake", {}).get("full_name") else [],
                    case_insensitive_values=_case_insensitive_locked_values(context["injection_list"]),
                    allowed_extra_values=_allowed_adversarial_values(context),
                    min_words=context["length_plan"]["target_min_words"],
                    max_words=context["length_plan"]["target_max_words"],
                )
                if patched_val.ok or _acceptable_compact_fallback(patched_val):
                    text = patched_text
                    last_val = patched_val
                    break
        if "word_count_too_long" in last_val.reasons:
            shortened_text = get_backend().generate(
                build_prompt_shorten(context, text),
                context,
                max_new_tokens=_shorten_max_new_tokens(context),
                temperature=TEMPERATURE,
                top_p=TOP_P,
            )
            shortened_val = validate_text(
                shortened_text,
                locked,
                forbidden_values=context["content_plan"].get("forbidden_values", []),
                allowed_full_names=[context["profile_fake"]["full_name"]] if context.get("profile_fake", {}).get("full_name") else [],
                case_insensitive_values=_case_insensitive_locked_values(context["injection_list"]),
            allowed_extra_values=_allowed_adversarial_values(context),
                min_words=context["length_plan"]["target_min_words"],
                max_words=context["length_plan"]["target_max_words"],
            )
            if shortened_val.ok:
                text = shortened_text
                last_val = shortened_val
                break
            compacted_text = compact_overlong_text(context)
            compacted_val = validate_text(
                compacted_text,
                locked,
                forbidden_values=context["content_plan"].get("forbidden_values", []),
                allowed_full_names=[context["profile_fake"]["full_name"]] if context.get("profile_fake", {}).get("full_name") else [],
                case_insensitive_values=_case_insensitive_locked_values(context["injection_list"]),
            allowed_extra_values=_allowed_adversarial_values(context),
                min_words=context["length_plan"]["target_min_words"],
                max_words=context["length_plan"]["target_max_words"],
            )
            if compacted_val.ok:
                text = compacted_text
                last_val = compacted_val
                break
        if "word_count_too_short" in last_val.reasons:
            extended_text = ensure_min_words(text, context)
            extended_val = validate_text(
                extended_text,
                locked,
                forbidden_values=context["content_plan"].get("forbidden_values", []),
                allowed_full_names=[context["profile_fake"]["full_name"]] if context.get("profile_fake", {}).get("full_name") else [],
                case_insensitive_values=_case_insensitive_locked_values(context["injection_list"]),
                allowed_extra_values=_allowed_adversarial_values(context),
                min_words=context["length_plan"]["target_min_words"],
                max_words=context["length_plan"]["target_max_words"],
            )
            if extended_val.ok or acceptable_after_rule_repair(extended_val):
                text = extended_text
                last_val = extended_val
                break
        if last_val.missing_values:
            repaired_text = rule_repair_text(text, context, last_val)
            _fmt_fail = bool(set(last_val.reasons) & {"non_prose_markdown_or_bullet", "placeholder_present"})
            # DEBUG
            repaired_val = validate_text(
                repaired_text,
                locked,
                forbidden_values=context["content_plan"].get("forbidden_values", []),
                allowed_full_names=[context["profile_fake"]["full_name"]] if context.get("profile_fake", {}).get("full_name") else [],
                case_insensitive_values=_case_insensitive_locked_values(context["injection_list"]),
            allowed_extra_values=_allowed_adversarial_values(context),
                min_words=context["length_plan"]["target_min_words"],
                max_words=context["length_plan"]["target_max_words"],
            )
            _missing_accept = _acceptable_compact_fallback if _fmt_fail else acceptable_after_rule_repair
            if repaired_val.ok or _missing_accept(repaired_val):
                text = repaired_text
                last_val = repaired_val
                break
        if attempt == MAX_ATTEMPTS:
            repaired_text = rule_repair_text(text, context, last_val)
            _is_compact_repair = bool(set(last_val.reasons) & {"non_prose_markdown_or_bullet", "placeholder_present"})
            repaired_val = validate_text(
                repaired_text,
                locked,
                forbidden_values=context["content_plan"].get("forbidden_values", []),
                allowed_full_names=[context["profile_fake"]["full_name"]] if context.get("profile_fake", {}).get("full_name") else [],
                case_insensitive_values=_case_insensitive_locked_values(context["injection_list"]),
            allowed_extra_values=_allowed_adversarial_values(context),
                min_words=context["length_plan"]["target_min_words"],
                max_words=context["length_plan"]["target_max_words"],
            )
            _accept_fn = _acceptable_compact_fallback if _is_compact_repair else acceptable_after_rule_repair
            if repaired_val.ok or _accept_fn(repaired_val):
                text = repaired_text
                last_val = repaired_val
                break
            raise PipelineError(build_failure(context["record_id"], "B11_clean_validation", ";".join(last_val.reasons), attempt, context, text, last_val))
    _preannot_missing = [v for v in locked if v and v not in text]
    if _preannot_missing:
        text = repair_missing_locked_values(text, context, _preannot_missing)
    clean = text
    text = apply_optional_noise(clean, locked, rng)
    if validate_text(
        text,
        locked,
        forbidden_values=context["content_plan"].get("forbidden_values", []),
        allowed_full_names=[context["profile_fake"]["full_name"]] if context.get("profile_fake", {}).get("full_name") else [],
        case_insensitive_values=_case_insensitive_locked_values(context["injection_list"]),
            allowed_extra_values=_allowed_adversarial_values(context),
        min_words=context["length_plan"]["target_min_words"],
        max_words=context["length_plan"]["target_max_words"],
    ).hard_fail:
        text = clean
    spans = annotate_exact_spans(text, context["injection_list"]["fields"])
    span_errors = validate_spans(text, spans, context["injection_list"]["fields"])
    if span_errors:
        raise PipelineError(build_failure(context["record_id"], "B13_annotation", ";".join(span_errors), 1, context, text))
    density = density_metrics(spans, context["injection_list"]["fields"], text=text)
    is_hard_eval = context.get("format_plan", {}).get("difficulty") in {"hard", "adversarial", "eval_hard"}
    density_reasons = density_quality_reasons(density, hard_eval=is_hard_eval)
    density_warnings = density_warning_reasons(density, hard_eval=is_hard_eval)
    mapping_warnings = source_mapping_warnings(context["record_type_info"])
    if density_reasons:
        failure = build_failure(context["record_id"], "B13_density_validation", ";".join(density_reasons), 1, context, text)
        failure["density_metrics"] = density
        raise PipelineError(failure)
    diversity_ok, diversity_max_sim = _worker_local.diversity_pool.check(text)
    if not diversity_ok:
        raise PipelineError(build_failure(context["record_id"], "B12_diversity", f"too_similar:{diversity_max_sim:.3f}", 1, context, text))
    _worker_local.diversity_pool.add(text)
    return {
        "record_id": context["record_id"], "text": text, "spans": spans,
        "profile_fake": context["profile_fake"], "record_type_info": context["record_type_info"],
        "form_blank": context["form_blank"], "background_context": context["background_context"], "event_list": context["event_list"],
        "content_plan": context["content_plan"], "format_plan": context["format_plan"], "length_plan": context["length_plan"],
        "injection_list": context["injection_list"],
        "metadata": {"schema_version": "secureprep-record-v2", "run_id": run_id, "row_index": row_index, "model_source": "mock" if USE_MOCK_BACKEND else MODEL_NAME, "prompt_version": context["prompt_version"], **density, "density_warnings": density_warnings, "source_mapping_warnings": mapping_warnings, "diversity_max_similarity": diversity_max_sim, "created_at": utc_now_iso()},
    }

In [57]:
# CELL 8 - MAIN RUN LOOP
def effective_gpu_for_worker(shard_index, shard_count):
    if GPU_DEVICE_ID is not None and int(shard_count or 1) <= 1:
        return int(GPU_DEVICE_ID)
    if int(shard_count or 1) > 1:
        return int(shard_index)
    return None

def runtime_gpu_label(effective_gpu_id):
    if effective_gpu_id is not None:
        return f"gpu={int(effective_gpu_id) + 1}"
    try:
        import torch
        if torch.cuda.is_available():
            return f"gpu={torch.cuda.current_device() + 1}"
    except Exception:
        pass
    return "gpu=cpu"

def run_generation_worker(shard_index=None, shard_count=None, auto_worker=False):
    # Use local variables so concurrent threads don't overwrite each other's state.
    _shard_idx = int(shard_index) if shard_index is not None else PARALLEL_SHARD_INDEX
    _shard_cnt = int(shard_count) if shard_count is not None else PARALLEL_SHARD_COUNT
    _gpu_id = effective_gpu_for_worker(_shard_idx, _shard_cnt)
    print("GPU worker config:", "shard", _shard_idx, "of", _shard_cnt, "effective_gpu_id", _gpu_id)
    print_gpu_environment()
    # backend already injected via _backend_local when called from AUTO_GPU_WORKERS;
    # in single-GPU mode get_backend() lazy-loads on first generate call.

    rng = random.Random(SEED + PERSON_ID + _shard_idx * 100_000)
    model_safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", "mock" if USE_MOCK_BACKEND else MODEL_NAME.split("/")[-1])
    base_form_start = FORM_ID_START_OVERRIDE if FORM_ID_START_OVERRIDE is not None else PERSON_ID * FORMS_PER_ACCOUNT
    base_form_end = FORM_ID_END_OVERRIDE if FORM_ID_END_OVERRIDE is not None else (PERSON_ID + 1) * FORMS_PER_ACCOUNT
    form_start, form_end = split_range_for_parallel(base_form_start, base_form_end, _shard_cnt, _shard_idx)
    range_id = f"p{PERSON_ID}_s{_shard_idx}of{_shard_cnt}_f{form_start}-{form_end}_{model_safe}"
    run_id = range_id if OVERWRITE_SAME_RANGE_CHECKPOINT else f"{range_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    shard_name = range_id if OVERWRITE_SAME_RANGE_CHECKPOINT else run_id
    writer = CheckpointWriter(CHECKPOINT_DIR, run_id, shard_name)
    if OVERWRITE_SAME_RANGE_CHECKPOINT:
        writer.cleanup_old_range_files()
        writer.reset_for_fresh_run()
    _worker_local.diversity_pool = DiversityPool(max_size=100, threshold=0.90)
    start_time = time.time()
    n_ok = n_failed = 0
    gpu_label = runtime_gpu_label(_gpu_id)
    push_every = 0 if auto_worker else KAGGLE_PUSH_EVERY_ROWS

    print("RUN", run_id, gpu_label, "forms", form_start, form_end)
    for form_idx in range(form_start, min(form_end, len(raw_records))):
        raw_record = raw_records[form_idx]
        for repeat_idx in range(ROWS_PER_FORM):
            row_index = form_idx * ROWS_PER_FORM + repeat_idx
            record_id = f"{run_id}_row_{row_index:06d}"
            if writer.is_done(record_id):
                continue
            row_start = time.time()
            if time.time() - start_time > TIME_LIMIT_S:
                writer.save_worker()
                if not auto_worker:
                    push_checkpoint_async(f"timeout {run_id}")
                raise SystemExit
            try:
                record = generate_one(row_index, raw_record, rng, run_id)
                writer.write_record(record)
                n_ok += 1
                row_time = time.time() - row_start
                word_count = count_words(record["text"])
                length_plan = record.get("length_plan", {})
                target_range = f"{length_plan.get('target_min_words')}-{length_plan.get('target_max_words')}"
                print(f"[GPU {_gpu_id}] OK {record_id} time={row_time:.2f}s words={word_count} target={target_range} locked={len(record.get('injection_list', {}).get('locked_values', []))}")
                if PRINT_GENERATED_RECORDS and n_ok <= PRINT_RECORD_LIMIT:
                    print("GENERATED_RECORD_JSON")
                    print(json.dumps(record, ensure_ascii=False, indent=2)[:12000])
            except PipelineError as e:
                writer.write_failure(e.failure)
                n_failed += 1
                row_time = time.time() - row_start
                print(f"[GPU {_gpu_id}] FAIL", record_id, e.failure.get("stage"), e.failure.get("reason"), f"time={row_time:.2f}s")
                if e.failure.get("missing_values"):
                    print("missing_values:", e.failure.get("missing_values"))
                if e.failure.get("text_preview"):
                    print("text_preview:", e.failure.get("text_preview")[:1000])
            except Exception as e:
                ctx = {"record_type_info": build_record_type_info(raw_record, form_idx), "profile_fake": {"profile_id": f"profile_{row_index:06d}"}, "injection_list": {"fields": [], "locked_values": []}}
                tb = traceback.format_exc()
                writer.write_failure(build_failure(record_id, "UNCAUGHT", repr(e), 1, ctx, error_trace=tb[:3000]))
                n_failed += 1
                row_time = time.time() - row_start
                print(f"[GPU {_gpu_id}] ERROR", record_id, type(e).__name__, repr(e), f"time={row_time:.2f}s")
                print(tb[:1200])
                if FAIL_FAST_ON_ERROR:
                    writer.save_worker()
                    raise
            if CHECKPOINT_EVERY_ROWS and (n_ok + n_failed) % CHECKPOINT_EVERY_ROWS == 0:
                writer.save_worker()
                print("progress", "ok", len(writer.completed_ids), "failed", len(writer.failed_ids))
            if push_every and (n_ok + n_failed) % push_every == 0:
                push_checkpoint_async(f"progress {run_id} n={n_ok+n_failed}")

    writer.save_worker()
    if not auto_worker:
        push_checkpoint_to_kaggle(f"final {run_id}", wait=True)
    print("DONE", run_id, "ok", len(writer.completed_ids), "failed", len(writer.failed_ids))
    print("records:", writer.records_path)
    print("failures:", writer.failures_path)
    return {"run_id": run_id, "ok": len(writer.completed_ids), "failed": len(writer.failed_ids), "records": str(writer.records_path), "failures": str(writer.failures_path)}

if AUTO_GPU_WORKERS:
    import threading as _th
    worker_count = max(1, int(AUTO_GPU_WORKER_COUNT or 2))
    print("AUTO_GPU_WORKERS (threading)", worker_count, "forms", FORM_ID_START_OVERRIDE, FORM_ID_END_OVERRIDE)
    assert worker_count <= 2, "Kaggle 2xT4 supports at most 2 workers"
    # Load both models sequentially on the MAIN thread.
    # from_pretrained + BitsAndBytes is not safe to call from non-main threads,
    # so we pre-load here and inject into each worker's thread-local backend.
    _preloaded_backends = {}
    for _gi in range(worker_count):
        if not USE_MOCK_BACKEND:
            print(f"[main] Loading model on GPU {_gi}...")
            _preloaded_backends[_gi] = HFBackend(MODEL_NAME, LOAD_IN_4BIT, gpu_id=_gi)
            print(f"[main] GPU {_gi} ready")
        else:
            _preloaded_backends[_gi] = MockBackend()
    _thread_results = [None] * worker_count
    def _run_worker(shard_idx):
        _backend_local.backend = _preloaded_backends[shard_idx]  # inject pre-loaded model
        _thread_results[shard_idx] = run_generation_worker(shard_idx, worker_count, True)
    threads = [_th.Thread(target=_run_worker, args=(i,), daemon=True) for i in range(worker_count)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    push_checkpoint_to_kaggle("final auto gpu workers", wait=True)
    print("AUTO_GPU_WORKERS DONE", worker_count, _thread_results)
else:
    run_generation_worker()

AUTO_GPU_WORKERS (threading) 2 forms 2005 2025
[main] Loading model on GPU 0...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

[main] GPU 0 ready
[main] Loading model on GPU 1...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

[main] GPU 1 ready
GPU worker config: shard 0 of 2 effective_gpu_id 0
CUDA available: True visible_gpus: 2 names: ['Tesla T4', 'Tesla T4']
GPU worker config: shard 1 of 2 effective_gpu_id 1
CUDA available: True visible_gpus: 2 names: ['Tesla T4', 'Tesla T4']
RUN p0_s1of2_f2015-2025_gemma-3-4b-it gpu=2 forms 2015 2025
RUN p0_s0of2_f2005-2015_gemma-3-4b-it gpu=1 forms 2005 2015
[GPU 0] OK p0_s0of2_f2005-2015_gemma-3-4b-it_row_002005 time=79.74s words=301 target=300-408 locked=8
progress ok 1 failed 0
[GPU 1] OK p0_s1of2_f2015-2025_gemma-3-4b-it_row_002015 time=79.99s words=472 target=489-666 locked=15
progress ok 1 failed 0
[GPU 0] OK p0_s0of2_f2005-2015_gemma-3-4b-it_row_002006 time=108.72s words=612 target=500-666 locked=20
progress ok 2 failed 0
[GPU 1] OK p0_s1of2_f2015-2025_gemma-3-4b-it_row_002016 time=112.55s words=552 target=475-602 locked=19
progress ok 2 failed 0
[GPU 0] OK p0_s0of2_f2005-2015_gemma-3-4b-it_row_002007 time=124.65s words=601 target=575-575 locked=23
progress ok 

In [58]:
# CELL 9 - QUICK INSPECTION REPORT
model_safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", "mock" if USE_MOCK_BACKEND else MODEL_NAME.split("/")[-1])
base_form_start = FORM_ID_START_OVERRIDE if FORM_ID_START_OVERRIDE is not None else PERSON_ID * FORMS_PER_ACCOUNT
base_form_end = FORM_ID_END_OVERRIDE if FORM_ID_END_OVERRIDE is not None else (PERSON_ID + 1) * FORMS_PER_ACCOUNT
report_shard_count = int(AUTO_GPU_WORKER_COUNT if AUTO_GPU_WORKERS else PARALLEL_SHARD_COUNT)
report_paths = []
if "writer" in globals():
    report_paths.append(writer.records_path)
    report_name = shard_name
else:
    for shard_idx in range(report_shard_count):
        fs, fe = split_range_for_parallel(base_form_start, base_form_end, report_shard_count, shard_idx)
        range_id = f"p{PERSON_ID}_s{shard_idx}of{report_shard_count}_f{fs}-{fe}_{model_safe}"
        report_paths.append(Path(CHECKPOINT_DIR) / "shards" / f"{range_id}.jsonl")
    report_name = f"p{PERSON_ID}_auto{report_shard_count}_f{base_form_start}-{base_form_end}_{model_safe}"

records = []
for records_path in report_paths:
    if records_path.exists():
        with records_path.open("r", encoding="utf-8") as f:
            records.extend(json.loads(line) for line in f if line.strip())

field_counter = Counter()
span_errors = []
missing_locked = []
for record in records:
    text = record["text"]
    for span in record.get("spans", []):
        field_counter[span.get("field_name")] += 1
        if text[span["start"]:span["end"]] != span["text"]:
            span_errors.append((record["record_id"], span))
    for value in record.get("injection_list", {}).get("locked_values", []):
        if value not in text:
            missing_locked.append((record["record_id"], value))

report = {
    "n_records": len(records),
    "span_errors": len(span_errors),
    "missing_locked": len(missing_locked),
    "field_coverage": dict(field_counter),
    "created_at": utc_now_iso(),
}
report_path = Path(CHECKPOINT_DIR) / "reports" / f"report_{report_name}.json"
write_json_atomic(report_path, report)
print(json.dumps(report, ensure_ascii=False, indent=2))
print("report:", report_path)

{
  "n_records": 20,
  "span_errors": 0,
  "missing_locked": 0,
  "field_coverage": {
    "full_name": 47,
    "family_relations": 19,
    "marital_status": 20,
    "dob": 24,
    "address": 20,
    "health_status": 14,
    "sexual_orientation": 9,
    "bank_account": 20,
    "criminal_record": 11,
    "political_view": 9,
    "cccd": 17,
    "email": 19,
    "phone": 19,
    "religion": 8,
    "driver_license": 12,
    "passport_number": 12,
    "vehicle_plate": 10,
    "digital_account": 17,
    "location_data": 12,
    "eid_credentials": 13,
    "biometric": 7,
    "behavioral_data": 9,
    "private_life": 8,
    "ethnicity": 7
  },
  "created_at": "2026-05-28T17:47:44.279085+00:00"
}
report: /kaggle/working/checkpoints/reports/report_p0_auto2_f2005-2025_gemma-3-4b-it.json
